# SimpleDyG: bản sửa chạy tiếp từ Output và source manifest

Chạy từ đầu hai dataset UCI/ML-10M, 10 seed và ba biến thể temporal alignment với
`PROFILE="paper"`. Add Input ZIP source đi kèm rồi chọn **GPU T4 x2**.
Nên thử `PROFILE="smoke"` trong một phiên riêng trước phiên paper.

Hai worker lấy các run độc lập từ hàng đợi chung; mỗi worker chỉ nhìn thấy một GPU.
Giữ architecture, logical batch size, split, loss, protocol và chọn checkpoint bằng validation.
Baseline dùng gradient accumulation theo micro-batch (mặc định 8 event cho paper),
giữ logical batch 128; luồng dropout có thể khác bản baseline chạy một forward lớn.

`latest.pt` lưu model hiện tại, best model, optimizer, scheduler/scaler khi dùng,
RNG Python/NumPy/Torch/CUDA, trạng thái shuffle/negative sampler, epoch và batch cursor.
Sau khi đổi phiên, runtime khôi phục trạng thái thay vì khởi tạo optimizer mới.
Checkpoint cũ của notebook một GPU không tương thích định dạng mới; lịch này bắt đầu từ đầu.

Mặc định tạm dừng sau 10,5 giờ ở ranh giới batch hoặc evaluation để dành thời gian
đóng ZIP. Nếu bị tắt đột ngột, dùng latest.pt đã ghi đủ gần nhất; đoạn sau checkpoint
sẽ được tính lại. ZIP được ghi sang file tạm rồi mới thay snapshot hoàn chỉnh cũ.
Hai GPU không gộp VRAM cho một model; toàn bộ lịch 100 run có thể cần nhiều phiên.

Các giới hạn benchmark của bản trước vẫn áp dụng: BPR adapter từ DyGLib, zero features,
closed-world vocabulary và metric tương thích tác giả được ghi riêng. Kết quả smoke
chỉ kiểm tra thực thi, không dùng để kết luận reproduce.


**Bản chạy tiếp:** mặc định `RESUME=True`.
Add Input Output của phiên thành công gần nhất và chọn GPU T4 x2. Giữ ZIP source
cũ nếu có; khi cần clone repo, notebook vẫn đối chiếu inventory/checksum của Output.
Lỗi `Existing output has different config/core/source` do full clone có thêm file
được xử lý ở phần chuẩn bị source; checkpoint và training runtime không thay đổi.


In [ ]:
# Chạy tiếp Output mới nhất: chọn Kaggle Settings > Accelerator > GPU T4 x2.
PROFILE = "paper"                   # smoke | pilot | paper
PROTOCOL_OVERRIDE = None
DATASETS_OVERRIDE = None            # None = UCI và ML-10M
SEEDS_OVERRIDE = None               # None = 10 seed với paper
BASELINE_MODELS = ["TGAT", "GraphMixer"]
VARIANTS = ["full", "same_time", "no_time"]
EXTRA_CONFIG = {}                    # không đổi sau khi đã bắt đầu thí nghiệm

GPU_IDS = [0, 1]
SESSION_HOURS = 10.5                 # tạm dừng trước 12h để lưu Output/ZIP
CHECKPOINT_SECONDS = 300             # thêm checkpoint ở ranh giới mỗi epoch
WORKER_CPU_THREADS = 2
SNAPSHOT_INCLUDE_COMPLETED_BEST = True   # ZIP gồm cả best.pt của các run đã hoàn thành

RESUME = True                      # phiên đầu False; phiên tiếp theo True
RESUME_FROM = ""                    # ZIP/thư mục Output mới nhất; trống = tự tìm Input
OFFLINE = False
INSTALL_DEPENDENCIES = True
AUTHOR_ROOT_OVERRIDE = ""
DYGLIB_ROOT_OVERRIDE = ""
INPUT_BUNDLE_OVERRIDE = ""           # ZIP source SimpleDyG_Kaggle_2GPU_Inputs.zip


## 1. Môi trường và hai GPU

In [ ]:
import os, sys, subprocess, importlib.util
from pathlib import Path
os.environ["USE_TF"] = "0"
os.environ["USE_TORCH"] = "1"
WORK = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()/"kaggle_working"
WORK.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(WORK/"hf_cache")
os.environ["MPLCONFIGDIR"] = str(WORK/"mpl_cache")
requirements = {"boto3": "boto3>=1.34,<2", "matplotlib": "matplotlib>=3.7,<4",
                "requests": "requests>=2.31,<3", "filelock": "filelock>=3.12,<4",
                "tqdm": "tqdm>=4.65,<5"}
from importlib.metadata import version, PackageNotFoundError
needed = []
for name, requirement in requirements.items():
    try:
        installed = version(name)
    except PackageNotFoundError:
        needed.append(requirement)
if needed and INSTALL_DEPENDENCIES:
    cmd = [sys.executable, "-m", "pip", "install", "--quiet", "--only-binary=:all:"]
    if OFFLINE:
        wheels = sorted({str(p.parent) for p in Path("/kaggle/input").rglob("*.whl")})
        if not wheels:
            raise RuntimeError("Thiếu dependency khi offline. Add Input các wheel hoặc bật Internet rồi chạy lại setup.")
        cmd += ["--no-index"]
        for directory in wheels:
            cmd += ["--find-links", directory]
    subprocess.run(cmd + needed, check=True)
elif needed:
    raise RuntimeError(f"Thiếu/sai phiên bản dependency: {needed}")
import torch
from packaging.version import Version
if Version(torch.__version__.split("+")[0]) < Version("2.3"):
    raise RuntimeError("Pipeline cần PyTorch >=2.3 cho native AMP. Chọn image Kaggle mới; setup giữ PyTorch/CUDA có sẵn.")
print("Tokenizer node và legacy optimizer đã nhúng; không cài transformers/tokenizers, không build Rust.")
print("Python:", sys.version.split()[0], "| Torch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
if PROFILE != "smoke" and not torch.cuda.is_available():
    raise RuntimeError("Bật GPU Kaggle hoặc chọn PROFILE='smoke' để kiểm tra CPU.")

if PROFILE != "smoke" and torch.cuda.device_count() < 2:
    raise RuntimeError("Notebook này cần GPU T4 x2. Hãy chọn hai GPU trong Kaggle Settings.")
print("Visible GPUs:", torch.cuda.device_count())
for index in range(torch.cuda.device_count()):
    print("GPU", index, torch.cuda.get_device_name(index))


## 2. Runtime tự chứa cho hai tiến trình

In [ ]:
import json, zipfile, hashlib, importlib.util
import pandas as pd
DUAL_RUNTIME_SOURCE = '"""Source embedded verbatim in the standalone Kaggle notebook. No CLI side effects."""\nfrom pathlib import Path\nfrom collections import Counter, defaultdict\nimport contextlib\nimport copy\nimport gc\nimport hashlib\nimport importlib.util\nimport json\nimport math\nimport os\nimport random\nimport subprocess\nimport sys\nimport time\nimport types\nimport zipfile\n\nimport numpy as np\nimport pandas as pd\nos.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")\nimport torch\nfrom torch import nn\nfrom torch.utils.data import DataLoader\n\n\ndef gelu_new(x):\n    """GPT-2 tanh GELU, same arithmetic as transformers.activations.NewGELUActivation."""\n    return 0.5*x*(1.0+torch.tanh(math.sqrt(2.0/math.pi)*(x+0.044715*torch.pow(x, 3.0))))\n\n\ndef get_activation(name):\n    activations = {\'gelu_new\': gelu_new, \'gelu\': torch.nn.functional.gelu,\n                   \'relu\': torch.nn.functional.relu, \'tanh\': torch.tanh,\n                   \'sigmoid\': torch.sigmoid, \'silu\': torch.nn.functional.silu,\n                   \'swish\': torch.nn.functional.silu}\n    if name not in activations:\n        raise ValueError(f\'Activation outside the released GPT-2 path: {name}\')\n    return activations[name]\n\n\nclass AuthorAdamW(torch.optim.Optimizer):\n    """Legacy HF AdamW arithmetic, including epsilon before bias correction.\n\n    Reference: Hugging Face transformers v4.44.2 optimization.py (Apache-2.0).\n    Embedded to keep author_compat independent of the optional HF/Rust packages.\n    """\n    def __init__(self, params, lr=1e-3, betas=(.9, .999), eps=1e-8,\n                 weight_decay=0., correct_bias=True):\n        if lr < 0 or eps < 0 or weight_decay < 0 or any(not 0 <= b < 1 for b in betas):\n            raise ValueError(\'Invalid AdamW parameters\')\n        super().__init__(params, dict(lr=lr, betas=betas, eps=eps,\n                                      weight_decay=weight_decay, correct_bias=correct_bias))\n\n    @torch.no_grad()\n    def step(self, closure=None):\n        loss = None\n        if closure is not None:\n            with torch.enable_grad():\n                loss = closure()\n        for group in self.param_groups:\n            beta1, beta2 = group[\'betas\']\n            for parameter in group[\'params\']:\n                if parameter.grad is None:\n                    continue\n                gradient = parameter.grad\n                if gradient.is_sparse:\n                    raise RuntimeError(\'AuthorAdamW requires dense gradients\')\n                state = self.state[parameter]\n                if not state:\n                    state.update(step=0, exp_avg=torch.zeros_like(parameter), exp_avg_sq=torch.zeros_like(parameter))\n                state[\'step\'] += 1\n                m, v = state[\'exp_avg\'], state[\'exp_avg_sq\']\n                m.mul_(beta1).add_(gradient, alpha=1.-beta1)\n                v.mul_(beta2).addcmul_(gradient, gradient, value=1.-beta2)\n                denominator = v.sqrt().add_(group[\'eps\'])\n                step_size = group[\'lr\']\n                if group[\'correct_bias\']:\n                    step_size = step_size*math.sqrt(1.-beta2**state[\'step\'])/(1.-beta1**state[\'step\'])\n                parameter.addcdiv_(m, denominator, value=-step_size)\n                if group[\'weight_decay\']:\n                    parameter.add_(parameter, alpha=-group[\'lr\']*group[\'weight_decay\'])\n        return loss\n\n\ndef linear_warmup_decay(optimizer, warmup_steps, total_steps):\n    def scale(step):\n        if step < warmup_steps:\n            return float(step)/max(1, warmup_steps)\n        return max(0., float(total_steps-step)/max(1, total_steps-warmup_steps))\n    return torch.optim.lr_scheduler.LambdaLR(optimizer, scale)\n\nAUTHOR_COMMIT = \'fa0dc82587d8ce787345a0379f65a0dcf5d9d9ff\'\nDYGLIB_COMMIT = \'3aacc36b94b8d2d8293d70a74fdf6d39089b4163\'\nSPECS = {\n    \'UCI_13\': dict(raw=\'uci\', timestamp=12, layers=6, heads=8, dim=768, batch=32, lr=1e-4),\n    \'ML_10M_13\': dict(raw=\'movie\', timestamp=12, layers=2, heads=4, dim=128, batch=64, lr=1e-5),\n}\nPAPER = {\n    \'UCI_13\': {\'full\': (.104, .010, .092, .014), \'same_time\': (.090, .013, .083, .012), \'no_time\': (.111, .015, .091, .014)},\n    \'ML_10M_13\': {\'full\': (.138, .009, .131, .008), \'same_time\': (.147, .001, .139, .001), \'no_time\': (.117, .062, .111, .059)},\n}\n\n\ndef make_config(profile=\'pilot\', **overrides):\n    if profile not in {\'smoke\', \'pilot\', \'paper\'}:\n        raise ValueError(profile)\n    cfg = dict(profile=profile, datasets=[\'UCI_13\', \'ML_10M_13\'], seeds=[0, 1, 2],\n               protocol=\'corrected\', variants=[\'full\', \'same_time\', \'no_time\'],\n               baseline_models=[\'TGAT\', \'GraphMixer\'], baseline_dims=[64, 128],\n               epochs=100, baseline_epochs=50, patience=10, train_block=512,\n               context=1024, max_new_tokens=64, micro_batch=8, amp=True,\n               baseline_batch=128, score_batch=64, history_cap=128,\n               mixer_tokens=64, tgat_neighbors=20, time_unit_seconds=86400.,\n               train_limit=None, eval_limit=None, edge_limit=None,\n               export_checkpoints=False, device=\'auto\', baseline_micro_batch=8, execution_format=\'dual_resume_v1\')\n    if profile == \'smoke\':\n        cfg.update(datasets=[\'UCI_13\'], seeds=[0], epochs=1, baseline_epochs=1,\n                   patience=1, train_block=64, context=128, max_new_tokens=8,\n                   micro_batch=4, baseline_batch=16, score_batch=64,\n                   baseline_dims=[16], baseline_micro_batch=16, history_cap=8, mixer_tokens=8,\n                   tgat_neighbors=4, train_limit=32, eval_limit=4, edge_limit=64, amp=False)\n    if profile == \'paper\':\n        cfg.update(seeds=list(range(10)), protocol=\'author_compat\', amp=False,\n                   baseline_epochs=100, baseline_dims=[16, 32, 64, 128, 256, 512],\n                   history_cap=1024, mixer_tokens=1024)\n    cfg.update(overrides)\n    if not cfg[\'seeds\'] or not cfg[\'datasets\'] or not (cfg[\'variants\'] or cfg[\'baseline_models\']):\n        raise ValueError(\'At least one dataset, seed and model/variant is required\')\n    if any(key not in {\'TGAT\', \'GraphMixer\', \'DyGFormer\'} for key in cfg[\'baseline_models\']):\n        raise ValueError(\'Unsupported baseline name\')\n    if any(v not in {\'full\', \'same_time\', \'no_time\', \'per_ego_permute\'} for v in cfg[\'variants\']):\n        raise ValueError(\'Unsupported temporal variant\')\n    if any(d < 2 or d % 2 for d in cfg[\'baseline_dims\']):\n        raise ValueError(\'Baseline dimensions must be positive even integers\')\n    if cfg[\'time_unit_seconds\'] <= 0 or cfg[\'history_cap\'] < 2 or cfg[\'mixer_tokens\'] < 2:\n        raise ValueError(\'Invalid history/time configuration\')\n    if not isinstance(cfg[\'baseline_micro_batch\'], int) or cfg[\'baseline_micro_batch\'] < 1:\n        raise ValueError(\'baseline_micro_batch must be a positive integer\')\n    if cfg[\'protocol\'] not in {\'corrected\', \'author_compat\'}:\n        raise ValueError(\'Unknown protocol\')\n    if cfg[\'protocol\'] == \'author_compat\' and cfg[\'context\'] != 1024:\n        raise ValueError(\'author_compat requires the released 1024-position architecture\')\n    if cfg[\'train_block\'] > cfg[\'context\'] or cfg[\'max_new_tokens\'] >= cfg[\'context\']:\n        raise ValueError(\'Invalid sequence limits\')\n    if any(ds not in SPECS for ds in cfg[\'datasets\']):\n        raise ValueError(\'This pipeline supports the complete UCI and ML-10M releases only\')\n    return cfg\n\n\ndef dump_json(path, value):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = path.with_suffix(path.suffix + \'.tmp\')\n    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False), encoding=\'utf-8\')\n    temporary.replace(path)\n\n\ndef sha256(path):\n    h = hashlib.sha256()\n    with Path(path).open(\'rb\') as f:\n        for block in iter(lambda: f.read(1024 * 1024), b\'\'):\n            h.update(block)\n    return h.hexdigest()\n\n\ndef seed_all(seed):\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(seed)\n    torch.backends.cudnn.benchmark = False\n    torch.backends.cudnn.deterministic = True\n\n\ndef read_lines(path):\n    return [x.split() for x in Path(path).read_text(encoding=\'utf-8\').splitlines() if x.strip()]\n\n\ndef unique(xs):\n    return list(dict.fromkeys(xs))\n\n\ndef metrics(pred, truth, author=False):\n    """Standard binary NDCG deduplicates both lists; author metric preserves duplicates."""\n    p, y = (list(pred), list(truth)) if author else (unique(pred), unique(truth))\n    if not y:\n        raise ValueError(\'Empty ground truth must be explicitly excluded before scoring\')\n    denom = sum(1 / math.log2(i + 2) for i in range(min(5, len(y))))\n    ndcg = sum(1 / math.log2(i + 2) for i, x in enumerate(p[:5]) if x in y) / denom\n    a, b = set(pred), set(truth)\n    jac = len(a & b) / len(a | b)\n    return {\'ndcg5\': ndcg, \'jaccard\': jac}\n\n\ndef summarize_predictions(rows, out_path=None):\n    eligible = [r for r in rows if r[\'truth\']]\n    if not eligible:\n        raise ValueError(\'No eligible evaluation queries\')\n    result = {}\n    for key in [\'ndcg5\', \'ndcg5_author\', \'jaccard\', \'jaccard_at5\']:\n        result[key] = float(np.mean([r[key] for r in eligible]))\n    result.update(n_total=len(rows), n_evaluated=len(eligible), n_empty_target=len(rows)-len(eligible),\n                  ended_fraction=float(np.mean([r.get(\'ended\', True) for r in eligible])))\n    if out_path is not None:\n        with Path(out_path).open(\'w\', encoding=\'utf-8\') as f:\n            for r in rows:\n                f.write(json.dumps(r, ensure_ascii=False) + \'\\n\')\n    return result\n\n\ndef load_dyglib(root):\n    """Namespace-only import rewriting prevents clashes with the author\'s models/utils."""\n    base = \'dyglib_vendor\'\n    for suffix, folder in [(\'\', root), (\'.models\', Path(root)/\'models\'), (\'.utils\', Path(root)/\'utils\')]:\n        name = base+suffix\n        if name not in sys.modules:\n            mod = types.ModuleType(name)\n            mod.__path__ = [str(folder)]\n            sys.modules[name] = mod\n    for rel in [\'utils/DataLoader.py\', \'utils/utils.py\', \'models/modules.py\',\n                \'models/TGAT.py\', \'models/GraphMixer.py\', \'models/DyGFormer.py\']:\n        name = base+\'.\'+rel[:-3].replace(\'/\', \'.\')\n        if name in sys.modules:\n            continue\n        file = Path(root)/rel\n        source = file.read_text(encoding=\'utf-8\').replace(\'from models.\', f\'from {base}.models.\').replace(\'from utils.\', f\'from {base}.utils.\')\n        mod = types.ModuleType(name)\n        mod.__file__ = str(file)\n        sys.modules[name] = mod\n        exec(compile(source, str(file), \'exec\'), mod.__dict__)\n    return {key: getattr(sys.modules[base+\'.models.\'+key], key) for key in [\'TGAT\', \'GraphMixer\', \'DyGFormer\']}, sys.modules[base+\'.utils.utils\'].NeighborSampler\n\n\ndef build_sampler(ds, cfg, Sampler):\n    # Only events strictly before the test bin may enter the neighbor store.\n    # Validation and training further filter by exact query time using searchsorted.\n    maximum = ds[\'spec\'][\'timestamp\']\n    observed = ds[\'raw\'][ds[\'raw\'].timestamp < maximum]\n    adjacency = [[] for _ in range(len(ds[\'vocab\'])+1)]\n    for row in observed.itertuples():\n        u, v = ds[\'vocab\'][str(row.u)]+1, ds[\'vocab\'][str(row.i)]+1\n        t = float(row.ts)/cfg[\'time_unit_seconds\']\n        adjacency[u].append((v, int(row.idx), t))\n        adjacency[v].append((u, int(row.idx), t))\n    class CappedSampler(Sampler):\n        def find_neighbors_before(self, node_id, interact_time, return_sampled_probabilities=False):\n            values = super().find_neighbors_before(node_id, interact_time, return_sampled_probabilities)\n            return tuple(x[-cfg[\'history_cap\']:] if x is not None else None for x in values)\n    sampler = CappedSampler(adjacency, sample_neighbor_strategy=\'recent\', seed=0)\n    # Executable no-future-history check at both frozen evaluation cutoffs.\n    for bin_id in [maximum-1, maximum]:\n        cutoff = float(ds[\'raw\'].loc[ds[\'raw\'].timestamp == bin_id, \'ts\'].min())/cfg[\'time_unit_seconds\']\n        for u in range(1, len(adjacency)):\n            _, _, times, _ = sampler.find_neighbors_before(u, cutoff)\n            assert np.all(times < cutoff)\n    return sampler\n\n\nclass TemporalRanker(nn.Module):\n    def __init__(self, key, dim, ds, cfg, device, sampler, classes):\n        super().__init__()\n        self.key, self.cfg = key, cfg\n        # UCI release contains all-zero features; ML-10M has no aligned .npy.\n        # Both baselines use explicit zero features. No invented semantic features.\n        nodes = np.zeros((len(ds[\'vocab\'])+1, dim), dtype=np.float32)\n        edges = np.zeros((int(ds[\'raw\'].idx.max())+1, dim), dtype=np.float32)\n        common = dict(node_raw_features=nodes, edge_raw_features=edges, neighbor_sampler=sampler,\n                      time_feat_dim=dim, dropout=.1, device=str(device))\n        if key == \'TGAT\':\n            self.backbone = classes[key](**common, num_layers=1, num_heads=2)\n        elif key == \'GraphMixer\':\n            self.backbone = classes[key](**common, num_tokens=cfg[\'mixer_tokens\'],\n                                         num_layers=1 if ds[\'name\']==\'UCI_13\' else 2)\n        elif key == \'DyGFormer\':\n            self.backbone = classes[key](**common, channel_embedding_dim=dim, patch_size=2,\n                                         num_layers=2, num_heads=2,\n                                         max_input_sequence_length=cfg[\'history_cap\'])\n        else:\n            raise ValueError(key)\n        self.head = nn.Sequential(nn.Linear(2*dim, dim), nn.ReLU(), nn.Linear(dim, 1))\n        self.to(device)\n\n    def forward(self, src, dst, times):\n        kw = dict(src_node_ids=src, dst_node_ids=dst, node_interact_times=times)\n        if self.key == \'TGAT\':\n            kw[\'num_neighbors\'] = self.cfg[\'tgat_neighbors\']\n        if self.key == \'GraphMixer\':\n            kw.update(num_neighbors=self.cfg[\'mixer_tokens\'], time_gap=self.cfg[\'history_cap\'])\n        a, b = self.backbone.compute_src_dst_node_temporal_embeddings(**kw)\n        return self.head(torch.cat([a, b], dim=-1)).squeeze(-1)\n\n    def embed(self, node_ids, times):\n        if self.key == \'TGAT\':\n            return self.backbone.compute_node_temporal_embeddings(node_ids, times,\n                current_layer_num=self.backbone.num_layers, num_neighbors=self.cfg[\'tgat_neighbors\'])\n        if self.key == \'GraphMixer\':\n            return self.backbone.compute_node_temporal_embeddings(node_ids, times,\n                num_neighbors=self.cfg[\'mixer_tokens\'], time_gap=self.cfg[\'history_cap\'])\n        raise ValueError(\'DyGFormer embeddings depend on the node pair and cannot be cached separately\')\n\n\n@torch.no_grad()\ndef evaluate_ranker(model, ds, split, cfg, device, k_choice=None, output=None):\n    model.eval()\n    target_bin = ds[\'spec\'][\'timestamp\'] - (split == \'val\')\n    cutoff = float(ds[\'raw\'].loc[ds[\'raw\'].timestamp == target_bin, \'ts\'].min())/cfg[\'time_unit_seconds\']\n    candidates = np.arange(1, len(ds[\'vocab\'])+1, dtype=np.int64)\n    inverse = {v+1: k for k, v in ds[\'vocab\'].items()}\n    # TGAT/GraphMixer have independent node embeddings, so full-candidate evaluation\n    # can cache them at one frozen cutoff. DyGFormer needs full pairwise scoring.\n    cached = None\n    if model.key != \'DyGFormer\':\n        cached = torch.cat([model.embed(ids, np.full(len(ids), cutoff))\n                            for ids in np.array_split(candidates, math.ceil(len(candidates)/cfg[\'score_batch\']))])\n    rows, per_k = [], {k: [] for k in [1, 5, 10, 20]}\n    for inp, target in zip(ds[split], ds[split+\'_gt\']):\n        check_deadline()\n        ego = inp[2]\n        u = ds[\'vocab\'][ego]+1\n        truth = [x for x in target if x in ds[\'vocab\'] and x != ego]\n        parts = []\n        for start in range(0, len(candidates), cfg[\'score_batch\']):\n            check_deadline()\n            ids = candidates[start:start+cfg[\'score_batch\']]\n            if cached is None:\n                scores = model(np.full(len(ids), u), ids, np.full(len(ids), cutoff))\n            else:\n                src_embedding = cached[u-1].expand(len(ids), -1)\n                scores = model.head(torch.cat([src_embedding, cached[start:start+len(ids)]], dim=-1)).squeeze(-1)\n            parts.append(scores.cpu().numpy())\n        scores = np.concatenate(parts)\n        if not np.isfinite(scores).all():\n            raise FloatingPointError(\'Non-finite candidate scores\')\n        scores[u-1] = -np.inf\n        # Stable ties are broken by canonical vocabulary IDs, equally for all baselines.\n        ranked = [inverse[int(i+1)] for i in np.argsort(-scores, kind=\'stable\')[:20]]\n        if truth:\n            for k in per_k:\n                check_deadline()\n                per_k[k].append(metrics(ranked[:k], truth)[\'jaccard\'])\n        rows.append(dict(ego=ego, ranking=ranked, truth=truth))\n    if not any(per_k.values()):\n        raise ValueError(\'No eligible ranking queries\')\n    macro_k = {k: float(np.mean(v)) for k, v in per_k.items()}\n    # Select output cardinality on validation only. The paper\'s test max-over-k\n    # is reported separately as an oracle reference, never as the primary score.\n    selected = max(macro_k, key=lambda k: (macro_k[k], -k)) if k_choice is None else k_choice\n    predictions = [prediction_row(r[\'ego\'], r[\'ranking\'][:selected], r[\'truth\'], ranking=r[\'ranking\']) for r in rows]\n    for r, original in zip(predictions, rows):\n        check_deadline()\n        if r[\'truth\']:\n            r[\'ndcg5\'] = metrics(original[\'ranking\'], r[\'truth\'])[\'ndcg5\']\n            r[\'ndcg5_author\'] = metrics(original[\'ranking\'], r[\'truth\'], author=True)[\'ndcg5\']\n            r[\'jaccard_at5\'] = metrics(original[\'ranking\'][:5], r[\'truth\'])[\'jaccard\']\n    result = summarize_predictions(predictions, output)\n    result.update(selected_k=selected, jaccard_oracle_max_k=max(macro_k.values()),\n                  jaccard_by_k=macro_k, candidate_count=len(candidates)-1)\n    return result\n\n\ndef baseline_training_edges(ds, cfg):\n    train = ds[\'raw\'][ds[\'raw\'].timestamp <= ds[\'spec\'][\'timestamp\']-2].copy()\n    forbidden = defaultdict(set)\n    # All positives in each TRAIN bin are labels, never validation/test labels.\n    for r in train.itertuples():\n        u, v = ds[\'vocab\'][str(r.u)]+1, ds[\'vocab\'][str(r.i)]+1\n        forbidden[(u, int(r.timestamp))].update([u, v])\n        forbidden[(v, int(r.timestamp))].update([v, u])\n    pairs = [(ds[\'vocab\'][str(r.u)]+1, ds[\'vocab\'][str(r.i)]+1,\n              float(r.ts)/cfg[\'time_unit_seconds\'], int(r.timestamp)) for r in train.itertuples()]\n    pairs += [(v, u, t, b) for u, v, t, b in pairs]\n    if cfg[\'edge_limit\']:\n        pairs = pairs[:cfg[\'edge_limit\']]\n    return pairs, forbidden\n\n\ndef train_ranker_candidate(key, dim, seed, ds, cfg, device, sampler, classes, run_dir):\n    seed_all(seed)\n    model = TemporalRanker(key, dim, ds, cfg, device, sampler, classes)\n    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)\n    pairs, forbidden = baseline_training_edges(ds, cfg)\n    rng = np.random.default_rng(seed)\n    best, stale, best_epoch, history = -float(\'inf\'), 0, 0, []\n    checkpoint = run_dir/f\'candidate_dim{dim}.pt\'\n    for epoch in range(1, cfg[\'baseline_epochs\']+1):\n        model.train()\n        losses = []\n        for start in range(0, len(pairs), cfg[\'baseline_batch\']):\n            # Chronological batches; models here have no stateful node memory.\n            batch = pairs[start:start+cfg[\'baseline_batch\']]\n            src = np.array([r[0] for r in batch], dtype=np.int64)\n            dst = np.array([r[1] for r in batch], dtype=np.int64)\n            times = np.array([r[2] for r in batch], dtype=np.float64)\n            neg = []\n            for u, _, _, bin_id in batch:\n                blocked = forbidden[(u, bin_id)]\n                if len(blocked) >= len(ds[\'vocab\']):\n                    raise ValueError(\'No negative candidate\')\n                item = int(rng.integers(1, len(ds[\'vocab\'])+1))\n                while item in blocked:\n                    item = int(rng.integers(1, len(ds[\'vocab\'])+1))\n                neg.append(item)\n            optimizer.zero_grad(set_to_none=True)\n            # One joint call uses a common realization of dropout for the BPR batch.\n            scores = model(np.concatenate([src, src]), np.concatenate([dst, np.array(neg)]), np.concatenate([times, times]))\n            pos, negative = scores.chunk(2)\n            loss = torch.nn.functional.softplus(negative-pos).mean()\n            if not torch.isfinite(loss):\n                raise FloatingPointError(\'Non-finite BPR loss\')\n            loss.backward()\n            nn.utils.clip_grad_norm_(model.parameters(), 1.)\n            optimizer.step()\n            losses.append(float(loss.detach().cpu()))\n        validation = evaluate_ranker(model, ds, \'val\', cfg, device)\n        score = validation[\'ndcg5\']\n        if score > best + 1e-12:\n            best, stale, best_epoch = score, 0, epoch\n            torch.save(model.state_dict(), checkpoint)\n        else:\n            stale += 1\n        history.append(dict(epoch=epoch, train_loss=float(np.mean(losses)), **validation))\n        dump_json(run_dir/f\'history_dim{dim}.json\', history)\n        print(f"{ds[\'name\']} {key} d={dim} seed={seed} epoch={epoch} val_NDCG5={score:.4f} stale={stale}", flush=True)\n        if stale >= cfg[\'patience\']:\n            break\n    model.load_state_dict(torch.load(checkpoint, map_location=device, weights_only=True))\n    validation = evaluate_ranker(model, ds, \'val\', cfg, device)\n    return model, validation, best_epoch, len(history), checkpoint\n\n\ndef train_ranker(key, seed, ds, cfg, device, dyglib, run_dir):\n    classes, Sampler = load_dyglib(dyglib)\n    sampler = build_sampler(ds, cfg, Sampler)\n    began = time.perf_counter()\n    choices = []\n    for dim in cfg[\'baseline_dims\']:\n        model, validation, epoch, epochs_run, checkpoint = train_ranker_candidate(key, dim, seed, ds, cfg, device, sampler, classes, run_dir)\n        choices.append(dict(dim=dim, validation=validation, best_epoch=epoch,\n                            epochs_run=epochs_run, checkpoint=str(checkpoint)))\n        del model\n        gc.collect()\n        if device.type == \'cuda\':\n            torch.cuda.empty_cache()\n    winner = max(choices, key=lambda c: (c[\'validation\'][\'ndcg5\'], -c[\'dim\']))\n    dump_json(run_dir/\'validation_search.json\', choices)\n    model = TemporalRanker(key, winner[\'dim\'], ds, cfg, device, sampler, classes)\n    model.load_state_dict(torch.load(winner[\'checkpoint\'], map_location=device, weights_only=True))\n    validation = evaluate_ranker(model, ds, \'val\', cfg, device, output=run_dir/\'val_predictions.jsonl\')\n    training_seconds = time.perf_counter()-began\n    testing = time.perf_counter()\n    test = evaluate_ranker(model, ds, \'test\', cfg, device, k_choice=validation[\'selected_k\'], output=run_dir/\'test_predictions.jsonl\')\n    torch.save(model.state_dict(), run_dir/\'best.pt\')\n    # Candidate checkpoints are local intermediates; retain only the chosen checkpoint.\n    for c in choices:\n        Path(c[\'checkpoint\']).unlink()\n    result = dict(model=key, variant=\'full\', seed=seed, architecture={\'dim\': winner[\'dim\']},\n                  best_epoch=winner[\'best_epoch\'], epochs_run=winner[\'epochs_run\'],\n                  parameters=sum(p.numel() for p in model.parameters()), validation=validation, test=test,\n                  training_seconds=training_seconds, testing_seconds=time.perf_counter()-testing,\n                  feature_policy=\'all-zero node/edge features; dimensions selected on validation\',\n                  training_task=\'chronological event BPR, both edge directions; frozen next-bin ranking at evaluation\')\n    del model\n    gc.collect()\n    if device.type == \'cuda\':\n        torch.cuda.empty_cache()\n    return result\n\n\ndef run_id(ds, model, variant, seed, cfg):\n    payload = dict(dataset=ds[\'name\'], model=model, variant=variant, seed=seed, config=cfg,\n                   data=ds[\'audit\'][\'files\'], author_commit=AUTHOR_COMMIT,\n                   dyglib_commit=DYGLIB_COMMIT, implementation=\'dual_resume_v1\')\n    digest = hashlib.sha256(json.dumps(payload, sort_keys=True).encode()).hexdigest()[:12]\n    return f"{ds[\'name\']}_{model}_{variant}_s{seed}_{digest}"\n\n\ndef execute_run(ds, model, variant, seed, cfg, device, output_root, author_root, dyglib_root):\n    identifier = run_id(ds, model, variant, seed, cfg)\n    directory = Path(output_root)/\'runs\'/identifier\n    directory.mkdir(parents=True, exist_ok=True)\n    result_path = directory/\'result.json\'\n    if result_path.exists():\n        previous = json.loads(result_path.read_text(encoding=\'utf-8\'))\n        if previous[\'status\'] == \'complete\':\n            print(\'Reuse completed run:\', identifier, flush=True)\n            return previous\n    dump_json(directory/\'config.json\', cfg)\n    if device.type == \'cuda\':\n        torch.cuda.reset_peak_memory_stats(device)\n    try:\n        if model == \'SimpleDyG\':\n            result = train_lm(author_root, ds, variant, seed, cfg, device, directory)\n        else:\n            result = train_ranker(model, seed, ds, cfg, device, dyglib_root, directory)\n        result.update(status=\'complete\', run_id=identifier, dataset=ds[\'name\'],\n                      protocol=cfg[\'protocol\'], profile=cfg[\'profile\'],\n                      truncated_data=bool(cfg[\'train_limit\'] or cfg[\'eval_limit\'] or cfg[\'edge_limit\']),\n                      peak_gpu_mb=float(torch.cuda.max_memory_allocated(device)/1e6) if device.type==\'cuda\' else None)\n    except Exception as exc:\n        import traceback\n        (directory/\'error.txt\').write_text(traceback.format_exc(), encoding=\'utf-8\')\n        result = dict(status=\'failed\', run_id=identifier, dataset=ds[\'name\'], model=model,\n                      variant=variant, seed=seed, error=str(exc), protocol=cfg[\'protocol\'], profile=cfg[\'profile\'])\n        print(\'FAILED:\', identifier, str(exc), flush=True)\n        gc.collect()\n        if device.type == \'cuda\':\n            torch.cuda.empty_cache()\n    dump_json(result_path, result)\n    return result\n\n\ndef export_results(records, cfg, output_root, make_archive=True):\n    root = Path(output_root)\n    dump_json(root/\'run_manifest.json\', records)\n    complete = [r for r in records if r[\'status\'] == \'complete\']\n    table = []\n    for r in complete:\n        table.append({k: r[k] for k in [\'run_id\', \'dataset\', \'model\', \'variant\', \'seed\', \'protocol\', \'profile\',\n                                      \'best_epoch\', \'epochs_run\', \'parameters\', \'training_seconds\', \'testing_seconds\', \'peak_gpu_mb\', \'worker_id\', \'physical_gpu\']} |\n                     {k: r[\'test\'][k] for k in [\'ndcg5\', \'ndcg5_author\', \'jaccard\', \'jaccard_at5\', \'n_evaluated\', \'ended_fraction\']} |\n                     {\'val_ndcg5\': r[\'validation\'][\'ndcg5\'], \'selected_k\': r[\'test\'].get(\'selected_k\'),\n                      \'jaccard_oracle_max_k\': r[\'test\'].get(\'jaccard_oracle_max_k\'), \'truncated_data\': r[\'truncated_data\']})\n    df = pd.DataFrame(table)\n    df.to_csv(root/\'results_per_seed.csv\', index=False)\n    summaries = []\n    if not df.empty:\n        for keys, g in df.groupby([\'dataset\', \'model\', \'variant\', \'protocol\', \'profile\']):\n            row = dict(zip([\'dataset\', \'model\', \'variant\', \'protocol\', \'profile\'], keys))\n            row[\'n_seeds\'] = g.seed.nunique()\n            for metric in [\'ndcg5\', \'ndcg5_author\', \'jaccard\', \'jaccard_at5\', \'training_seconds\', \'parameters\']:\n                row[metric+\'_mean\'] = g[metric].mean()\n                row[metric+\'_std\'] = g[metric].std(ddof=1)  # NaN for one seed: uncertainty is not zero.\n            summaries.append(row)\n    summary = pd.DataFrame(summaries)\n    summary.to_csv(root/\'summary_mean_std.csv\', index=False)\n    paper_rows = []\n    for dataset, variants in PAPER.items():\n        for variant, (n, ns, j, js) in variants.items():\n            paper_rows.append(dict(dataset=dataset, variant=variant, source=\'paper Table 2/4; not measured here\',\n                                   paper_ndcg5=n, paper_ndcg5_std=ns, paper_jaccard=j, paper_jaccard_std=js))\n    pd.DataFrame(paper_rows).to_csv(root/\'paper_reference.csv\', index=False)\n    comparisons = []\n    for row in summaries:\n        if row[\'model\'] != \'SimpleDyG\' or row[\'variant\'] not in PAPER[row[\'dataset\']]:\n            continue\n        reference = PAPER[row[\'dataset\']][row[\'variant\']]\n        measured_ndcg = row[\'ndcg5_author_mean\'] if cfg[\'protocol\']==\'author_compat\' else row[\'ndcg5_mean\']\n        comparisons.append(dict(dataset=row[\'dataset\'], variant=row[\'variant\'], protocol=cfg[\'protocol\'],\n            profile=cfg[\'profile\'], n_seeds=row[\'n_seeds\'], measured_ndcg5=measured_ndcg,\n            paper_ndcg5=reference[0], delta_ndcg5=measured_ndcg-reference[0],\n            measured_jaccard=row[\'jaccard_mean\'], paper_jaccard=reference[2],\n            delta_jaccard=row[\'jaccard_mean\']-reference[2],\n            interpretation=\'descriptive gap only; profile/protocol may differ; no reproduction success threshold\'))\n    pd.DataFrame(comparisons, columns=[\'dataset\', \'variant\', \'protocol\', \'profile\', \'n_seeds\',\n        \'measured_ndcg5\', \'paper_ndcg5\', \'delta_ndcg5\', \'measured_jaccard\', \'paper_jaccard\',\n        \'delta_jaccard\', \'interpretation\']).to_csv(root/\'paper_comparison.csv\', index=False)\n    paired = []\n    if not df.empty:\n        for dataset, g in df[df.model==\'SimpleDyG\'].groupby(\'dataset\'):\n            full = g[g.variant==\'full\'].set_index(\'seed\')\n            for variant in cfg[\'variants\']:\n                if variant == \'full\':\n                    continue\n                ablated = g[g.variant==variant].set_index(\'seed\')\n                common = full.index.intersection(ablated.index)\n                for metric in [\'ndcg5\', \'jaccard\', \'jaccard_at5\']:\n                    delta = ablated.loc[common, metric]-full.loc[common, metric]\n                    paired.append(dict(dataset=dataset, variant=variant, metric=metric, n_pairs=len(common),\n                                       delta_mean=delta.mean(), delta_std=delta.std(ddof=1)))\n    pd.DataFrame(paired, columns=[\'dataset\', \'variant\', \'metric\', \'n_pairs\', \'delta_mean\', \'delta_std\']).to_csv(root/\'ablation_paired_deltas.csv\', index=False)\n    try:\n        import matplotlib\n        matplotlib.use(\'Agg\')\n        import matplotlib.pyplot as plt\n        if not summary.empty:\n            for metric in [\'ndcg5\', \'jaccard_at5\']:\n                datasets = list(summary.dataset.unique())\n                fig, axes = plt.subplots(1, len(datasets), figsize=(7*len(datasets), 4), squeeze=False)\n                for ax, dataset in zip(axes[0], datasets):\n                    sub = summary[summary.dataset==dataset]\n                    labels = sub.model+\' / \'+sub.variant\n                    ax.bar(range(len(sub)), sub[metric+\'_mean\'], yerr=sub[metric+\'_std\'].fillna(0), capsize=4)\n                    ax.set_xticks(range(len(sub)), labels, rotation=35, ha=\'right\')\n                    ax.set_title(dataset+\' | \'+cfg[\'profile\'])\n                    ax.set_ylabel(metric)\n                    ax.grid(axis=\'y\', alpha=.2)\n                fig.tight_layout()\n                fig.savefig(root/f\'{metric}.png\', dpi=160)\n                plt.close(fig)\n    except ImportError:\n        print(\'Charts skipped: matplotlib unavailable\')\n    notes = [f"Profile: {cfg[\'profile\']}; protocol: {cfg[\'protocol\']}.",\n             f"Completed: {len(complete)}/{len(records)} runs. Expected seeds: {cfg[\'seeds\']}.",\n             \'Smoke/subset runs only validate execution and cannot support scientific claims.\',\n             \'NDCG5 and Jaccard@5 are the common comparison metrics; both deduplicate predictions/labels.\',\n             \'SimpleDyG Jaccard uses generated set; baseline Jaccard uses k selected on validation.\',\n             \'Baseline test max-over-k is a separate oracle reference; it is not used for model selection.\',\n             \'Author NDCG preserves duplicates and may exceed 1. Values are explicitly marked ndcg5_author.\',\n             \'Baseline history uses undirected raw edges; released sequence targets remain source-only.\',\n             \'Baselines are BPR adaptations from DyGLib, not a claim to exactly reproduce Table 2 baseline scores.\',\n             \'Pilot history limits and corrected padding/decoding differ from the original release.\',\n             \'Ten seeds do not guarantee agreement with the paper. Check data scope and protocol before interpreting gaps.\']\n    (root/\'READ_RESULTS.txt\').write_text(\'\\n\'.join(notes), encoding=\'utf-8\')\n    archive = root.parent/(root.name+\'.zip\')\n    if make_archive:\n        with zipfile.ZipFile(archive, \'w\', zipfile.ZIP_DEFLATED) as z:\n            for p in root.rglob(\'*\'):\n                if p.is_file() and (cfg[\'export_checkpoints\'] or p.suffix != \'.pt\'):\n                    z.write(p, p.relative_to(root))\n    return df, summary, archive\n\n\ndef run_pipeline(author_root, dyglib_root, cfg, output_root):\n    cfg = copy.deepcopy(cfg)\n    source_files = {}\n    for label, directory in [(\'author\', Path(author_root)/\'models\'), (\'dyglib\', Path(dyglib_root))]:\n        for file in sorted(directory.rglob(\'*.py\')):\n            source_files[label+\'/\'+str(file.relative_to(directory)).replace(\'\\\\\', \'/\')] = sha256(file)\n    cfg[\'_source_fingerprint\'] = hashlib.sha256(json.dumps(source_files, sort_keys=True).encode()).hexdigest()\n    cfg[\'_pipeline_fingerprint\'] = globals().get(\'PIPELINE_SOURCE_SHA256\', sha256(__file__) if \'__file__\' in globals() else \'notebook\')\n    root = Path(output_root)\n    root.mkdir(parents=True, exist_ok=True)\n    device = choose_device(cfg)\n    os.environ.setdefault(\'USE_TF\', \'0\')\n    os.environ.setdefault(\'USE_TORCH\', \'1\')\n    os.environ.setdefault(\'MPLCONFIGDIR\', str(root/\'mpl_cache\'))\n    torch.set_num_threads(min(4, os.cpu_count() or 1))\n    datasets = {name: load_dataset(Path(author_root), name, cfg) for name in cfg[\'datasets\']}\n    dump_json(root/\'data_audit.json\', {n: d[\'audit\'] for n, d in datasets.items()})\n    dump_json(root/\'config.json\', cfg)\n    dump_json(root/\'source_hashes.json\', source_files)\n    environment = dict(python=sys.version, torch=torch.__version__, numpy=np.__version__, pandas=pd.__version__,\n                       cuda=torch.version.cuda, device=str(device),\n                       gpu=torch.cuda.get_device_name(device) if device.type==\'cuda\' else None,\n                       author_expected_commit=AUTHOR_COMMIT, dyglib_expected_commit=DYGLIB_COMMIT)\n    import boto3\n    environment.update(boto3=boto3.__version__,\n                       tokenizer_runtime=\'embedded NodeTokenizer; no Hugging Face/Rust dependency\',\n                       author_optimizer=\'embedded legacy AdamW arithmetic\')\n    dump_json(root/\'environment.json\', environment)\n    (root/\'pip_freeze.txt\').write_text(subprocess.check_output([sys.executable, \'-m\', \'pip\', \'freeze\'], text=True), encoding=\'utf-8\')\n    records = []\n    for ds in datasets.values():\n        for variant in cfg[\'variants\']:\n            for seed in cfg[\'seeds\']:\n                records.append(execute_run(ds, \'SimpleDyG\', variant, seed, cfg, device, root, author_root, dyglib_root))\n                export_results(records, cfg, root)\n        for key in cfg[\'baseline_models\']:\n            for seed in cfg[\'seeds\']:\n                records.append(execute_run(ds, key, \'full\', seed, cfg, device, root, author_root, dyglib_root))\n                export_results(records, cfg, root)\n    df, summary, archive = export_results(records, cfg, root)\n    print(\'Outputs:\', root, \'\\nArchive:\', archive, flush=True)\n    failed = [r for r in records if r[\'status\']==\'failed\']\n    if failed:\n        raise RuntimeError(f"{len(failed)} runs failed; see run_manifest.json and each error.txt. No success claim was generated.")\n    return df, summary, archive\n\n\ndef prediction_row(ego, pred, truth, **extra):\n    if not truth:\n        return dict(ego=ego, pred=pred, truth=[], **extra)\n    m = metrics(pred, truth)\n    return dict(ego=ego, pred=pred, truth=truth, **m,\n                ndcg5_author=metrics(pred, truth, author=True)[\'ndcg5\'],\n                jaccard_at5=metrics(unique(pred)[:5], truth)[\'jaccard\'], **extra)\n\n\ndef transform_time(tokens, variant, ego, seed, maximum=12):\n    """Transforms labels only, preserving node chronology and split/target membership."""\n    times = {f\'<|time{i}|>\' for i in range(maximum + 1)}\n    if variant == \'full\':\n        return list(tokens)\n    if variant == \'no_time\':\n        return [x for x in tokens if x not in times]\n    if variant == \'same_time\':\n        return [\'<|time0|>\' if x in times else x for x in tokens]\n    if variant == \'per_ego_permute\':\n        key = int(hashlib.sha256(f\'{seed}:{ego}\'.encode()).hexdigest()[:16], 16)\n        labels = list(range(maximum + 1))\n        random.Random(key).shuffle(labels)\n        mapping = {f\'<|time{i}|>\': f\'<|time{j}|>\' for i, j in enumerate(labels)}\n        return [mapping.get(x, x) for x in tokens]\n    raise ValueError(variant)\n\n\nclass NodeTokenizer:\n    """Matches the author\'s WordLevel node IDs and order of added special tokens."""\n    def __init__(self, vocab, maximum):\n        self.vocab = dict(vocab)\n        self.nodes = set(vocab)\n        if sorted(vocab.values()) != list(range(len(vocab))):\n            raise ValueError(\'Node vocabulary is not contiguous\')\n        self.special = [\'<|endoftext|>\', \'<|history|>\', \'<|endofhistory|>\', \'<|pre|>\', \'<|endofpre|>\']\n        self.special += [f\'<|time{i}|>\' for i in range(maximum + 1)] + [\'[PAD]\']\n        for token in self.special:\n            self.vocab[token] = len(self.vocab)\n        self.inverse = {i: x for x, i in self.vocab.items()}\n        self.pad = self.vocab[\'[PAD]\']\n        self.eos = self.vocab[\'<|endoftext|>\']\n        self.endpred = self.vocab[\'<|endofpre|>\']\n\n    def encode(self, tokens):\n        return [self.vocab[x] for x in tokens]\n\n\ndef load_dataset(root, name, cfg):\n    spec = SPECS[name]\n    directory = Path(root) / \'resources\' / name / str(spec[\'timestamp\'])\n    vpath = Path(root) / \'vocabs\' / name / str(spec[\'timestamp\']) / \'vocab.json\'\n    files = {f: directory / f\'{f}.link_prediction\' for f in [\'train\', \'val\', \'val_gt\', \'test\', \'test_gt\']}\n    for p in [*files.values(), vpath]:\n        if not p.is_file():\n            raise FileNotFoundError(p)\n    vocab = json.loads(vpath.read_text())\n    data = {k: read_lines(p) for k, p in files.items()}\n    raw_path = Path(root) / \'all_data\' / spec[\'raw\'] / str(spec[\'timestamp\']) / f"ml_{spec[\'raw\']}.csv"\n    raw = pd.read_csv(raw_path).sort_values(\'ts\', kind=\'stable\').reset_index(drop=True)\n    assert {\'u\', \'i\', \'ts\', \'timestamp\', \'idx\'} <= set(raw.columns)\n    assert raw.timestamp.max() == spec[\'timestamp\']\n    assert raw[[\'u\', \'i\', \'ts\', \'timestamp\']].notna().all().all()\n    assert set(raw.u.astype(str)) | set(raw.i.astype(str)) == set(vocab)\n    bins = raw.groupby(\'timestamp\').ts.agg([\'min\', \'max\'])\n    assert all(bins.iloc[i][\'max\'] < bins.iloc[i+1][\'min\'] for i in range(len(bins)-1))\n    audit = {\'dataset\': name, \'nodes\': len(vocab), \'edges\': len(raw), \'splits\': {}, \'files\': {}}\n    for k, p in files.items():\n        audit[\'files\'][str(p.relative_to(root)).replace(\'\\\\\', \'/\')] = sha256(p)\n        for tokens in data[k]:\n            assert all(x in vocab or x.startswith(\'<|\') for x in tokens)\n    audit[\'files\'][str(vpath.relative_to(root)).replace(\'\\\\\', \'/\')] = sha256(vpath)\n    audit[\'files\'][str(raw_path.relative_to(root)).replace(\'\\\\\', \'/\')] = sha256(raw_path)\n    for split in [\'val\', \'test\']:\n        assert len(data[split]) == len(data[split + \'_gt\'])\n        expected = spec[\'timestamp\'] - (split == \'val\')\n        for inp, target in zip(data[split], data[split + \'_gt\']):\n            assert inp[:2] == [\'<|endoftext|>\', \'<|history|>\']\n            assert inp[-1] == \'<|endofhistory|>\'\n            assert target[:2] == [\'<|pre|>\', f\'<|time{expected}|>\']\n            hist_times = [int(x[6:-2]) for x in inp if x.startswith(\'<|time\')]\n            assert hist_times == sorted(hist_times) and max(hist_times, default=-1) < expected\n            actual = raw[(raw.timestamp == expected) & (raw.u == int(inp[2]))].i.astype(str).tolist()\n            assert Counter(actual) == Counter(x for x in target if x in vocab)\n        lens = [len(x) for x in data[split]]\n        audit[\'splits\'][split] = dict(queries=len(lens), max_length=max(lens),\n            above_context=sum(x > cfg[\'context\'] for x in lens),\n            duplicate_targets=sum(len([x for x in y if x in vocab]) != len(set(x for x in y if x in vocab)) for y in data[split+\'_gt\']))\n    for line in data[\'train\']:\n        before, after = line[:line.index(\'<|pre|>\')], line[line.index(\'<|pre|>\'):]\n        output_time = [int(x[6:-2]) for x in after if x.startswith(\'<|time\')]\n        assert len(output_time) == 1 and output_time[0] <= spec[\'timestamp\'] - 2\n        assert all(int(x[6:-2]) < output_time[0] for x in before if x.startswith(\'<|time\'))\n    train_lens = [len(x) for x in data[\'train\']]\n    audit[\'splits\'][\'train\'] = dict(queries=len(train_lens), max_length=max(train_lens),\n                                  truncated=sum(x > cfg[\'train_block\'] for x in train_lens))\n    # The released queries score original source nodes only; do not silently symmetrize targets.\n    audit[\'query_scope\'] = \'released source-node queries; raw graph used as undirected history by baselines\'\n    audit[\'closed_world_vocabulary\'] = \'all released node identities, including future identities; no future edges as history\'\n    if cfg[\'train_limit\']:\n        data[\'train\'] = data[\'train\'][:cfg[\'train_limit\']]\n    if cfg[\'eval_limit\']:\n        for split in [\'val\', \'test\']:\n            data[split] = data[split][:cfg[\'eval_limit\']]\n            data[split+\'_gt\'] = data[split+\'_gt\'][:cfg[\'eval_limit\']]\n    data.update(name=name, spec=spec, vocab=vocab, raw=raw, audit=audit)\n    return data\n\n\ndef load_author_model(root):\n    """Load released architecture; route its two HF activation imports to identical math."""\n    namespace = \'simpledyg_author_models\'\n    if namespace not in sys.modules:\n        path = Path(root) / \'models\'\n        module = types.ModuleType(namespace)\n        module.__path__ = [str(path)]\n        module.__package__ = namespace\n        module.__version__ = \'2.5.1\'\n        sys.modules[namespace] = module\n        activation_module = types.ModuleType(namespace+\'.runtime_activations\')\n        activation_module.gelu_new = gelu_new\n        activation_module.get_activation = get_activation\n        sys.modules[activation_module.__name__] = activation_module\n        try:\n            for stem in [\'file_utils\', \'configuration_utils\', \'configuration_gpt2\', \'modeling_utils\', \'modeling_gpt2\']:\n                file = path/(stem+\'.py\')\n                source = file.read_text(encoding=\'utf-8\')\n                source = source.replace(\'from transformers.activations import get_activation\',\n                                        \'from .runtime_activations import get_activation\')\n                source = source.replace(\'from transformers.activations import gelu_new\',\n                                        \'from .runtime_activations import gelu_new\')\n                name = namespace+\'.\'+stem\n                child = types.ModuleType(name)\n                child.__file__ = str(file)\n                child.__package__ = namespace\n                sys.modules[name] = child\n                exec(compile(source, str(file), \'exec\'), child.__dict__)\n                setattr(module, stem, child)\n            module.GPT2Config = sys.modules[namespace+\'.configuration_gpt2\'].GPT2Config\n            module.GPT2LMHeadModel = sys.modules[namespace+\'.modeling_gpt2\'].GPT2LMHeadModel\n        except Exception:\n            for name in list(sys.modules):\n                if name == namespace or name.startswith(namespace+\'.\'):\n                    del sys.modules[name]\n            raise\n    module = sys.modules[namespace]\n    return module.GPT2Config, module.GPT2LMHeadModel\n\n\ndef choose_device(cfg):\n    requested = cfg[\'device\']\n    device = torch.device(\'cuda:0\' if torch.cuda.is_available() else \'cpu\') if requested == \'auto\' else torch.device(requested)\n    if device.type == \'cpu\' and cfg[\'profile\'] != \'smoke\':\n        raise RuntimeError(\'Full training needs Kaggle GPU; select smoke for a CPU check\')\n    return device\n\n\ndef amp_context(cfg, device):\n    return torch.autocast(device_type=\'cuda\', dtype=torch.float16) if cfg[\'amp\'] and device.type == \'cuda\' else contextlib.nullcontext()\n\n\ndef make_lm(root, ds, cfg):\n    Config, Model = load_author_model(root)\n    params = {k: ds[\'spec\'][k] for k in [\'layers\', \'heads\', \'dim\']}\n    if cfg[\'profile\'] == \'smoke\':\n        params = dict(layers=2, heads=2, dim=64)\n    tok = NodeTokenizer(ds[\'vocab\'], ds[\'spec\'][\'timestamp\'])\n    # Match original initialize-50257-then-resize RNG behavior for author_compat.\n    c = Config(vocab_size=50257 if cfg[\'protocol\'] == \'author_compat\' else len(tok.vocab),\n               n_positions=cfg[\'context\'], n_ctx=cfg[\'context\'], n_embd=params[\'dim\'],\n               n_layer=params[\'layers\'], n_head=params[\'heads\'])\n    model = Model(c)\n    model.resize_token_embeddings(len(tok.vocab))\n    return model, tok, params\n\n\n@torch.no_grad()\ndef evaluate_lm(model, tok, ds, split, variant, seed, cfg, device, output=None):\n    model.eval()\n    rows = []\n    for inp, target in zip(ds[split], ds[split+\'_gt\']):\n        check_deadline()\n        ego = inp[2]\n        truth = [x for x in target if x in tok.nodes and x != ego]\n        tokens = transform_time(inp, variant, ego, seed, ds[\'spec\'][\'timestamp\'])\n        ids = tok.encode(tokens)\n        compat = cfg[\'protocol\'] == \'author_compat\'\n        if compat:\n            if len(ids) > cfg[\'context\']:\n                ids = ids[-1000:]\n            limit = 11 if split == \'val\' else max(1, cfg[\'context\'] - (len(tok.special)-1) - len(ids))\n            ids = ids[-(cfg[\'context\']-1):]\n        else:\n            limit = cfg[\'max_new_tokens\']\n            ids = ids[-(cfg[\'context\']-limit):]\n        limit = min(limit, cfg[\'context\']-len(ids))\n        generated, past, ended = [], None, False\n        tensor = torch.tensor([ids], device=device)\n        for _ in range(limit):\n            check_deadline()\n            with amp_context(cfg, device):\n                outputs = model(tensor, past=past)\n            idx = int(outputs[0][0, -1].argmax().item())\n            generated.append(idx)\n            past = outputs[1]\n            if idx == tok.eos or (not compat and idx == tok.endpred):\n                ended = True\n                break\n            tensor = torch.tensor([[idx]], device=device)\n        pred = [tok.inverse[x] for x in generated if tok.inverse[x] in tok.nodes and tok.inverse[x] != ego]\n        rows.append(prediction_row(ego, pred, truth, ended=ended, generated_tokens=len(generated)))\n    return summarize_predictions(rows, output)\n\n\ndef train_lm(root, ds, variant, seed, cfg, device, run_dir):\n    seed_all(seed)\n    model, tok, architecture = make_lm(root, ds, cfg)\n    model.to(device)\n    sequences = [tok.encode(transform_time(x, variant, x[2], seed, ds[\'spec\'][\'timestamp\']))[-cfg[\'train_block\']:] for x in ds[\'train\']]\n    compat = cfg[\'protocol\'] == \'author_compat\'\n    def collate(rows):\n        padded = torch.full((len(rows), max(map(len, rows))), tok.pad, dtype=torch.long)\n        for i, row in enumerate(rows):\n            padded[i, :len(row)] = torch.tensor(row)\n        return padded\n    logical_batch = ds[\'spec\'][\'batch\'] if cfg[\'profile\'] != \'smoke\' else cfg[\'micro_batch\']\n    micro = min(logical_batch, cfg[\'micro_batch\'])\n    if logical_batch % micro:\n        raise ValueError(\'micro_batch must divide the dataset logical batch size\')\n    generator = torch.Generator().manual_seed(seed)\n    loader = DataLoader(sequences, batch_size=logical_batch, shuffle=True, generator=generator,\n                        collate_fn=collate, drop_last=compat)\n    if len(loader) == 0:\n        raise ValueError(\'No training batches\')\n    if compat:\n        optimizer = AuthorAdamW(model.parameters(), lr=ds[\'spec\'][\'lr\'], eps=1e-8, weight_decay=0.)\n    else:\n        optimizer = torch.optim.AdamW(model.parameters(), lr=ds[\'spec\'][\'lr\'], eps=1e-8, weight_decay=0.)\n    scheduler = linear_warmup_decay(optimizer, 0, len(loader)*cfg[\'epochs\'])\n    scaler = torch.amp.GradScaler(\'cuda\', enabled=cfg[\'amp\'] and device.type == \'cuda\')\n    best, best_epoch, stale, history = -float(\'inf\'), 0, 0, []\n    checkpoint = run_dir / \'best.pt\'\n    began = time.perf_counter()\n    for epoch in range(1, cfg[\'epochs\']+1):\n        model.train()\n        losses = []\n        for batch in loader:\n            optimizer.zero_grad(set_to_none=True)\n            valid_total = (batch[:, 1:] != tok.pad).sum().item() if not compat else batch[:, 1:].numel()\n            for part in batch.split(micro):\n                inputs = part.to(device)\n                labels = inputs.clone()\n                if not compat:\n                    labels[labels == tok.pad] = -100\n                weight = ((labels[:, 1:] != -100).sum().item()) / valid_total\n                with amp_context(cfg, device):\n                    outputs = model(inputs, labels=labels, attention_mask=None if compat else inputs.ne(tok.pad))\n                    loss = outputs[0]\n                if not torch.isfinite(loss):\n                    raise FloatingPointError(\'Non-finite LM loss\')\n                scaler.scale(loss * weight).backward()\n                losses.append(float(loss.detach().cpu()))\n            scaler.unscale_(optimizer)\n            nn.utils.clip_grad_norm_(model.parameters(), 1.)\n            scaler.step(optimizer)\n            scaler.update()\n            scheduler.step()\n        validation = evaluate_lm(model, tok, ds, \'val\', variant, seed, cfg, device)\n        score = validation[\'ndcg5_author\' if compat else \'ndcg5\']\n        improved = score >= best if compat else score > best + 1e-12\n        if improved:\n            best, best_epoch, stale = score, epoch, 0\n            torch.save(model.state_dict(), checkpoint)\n        else:\n            stale += 1\n        history.append(dict(epoch=epoch, train_loss=float(np.mean(losses)), **validation))\n        dump_json(run_dir/\'history.json\', history)\n        print(f"{ds[\'name\']} SimpleDyG/{variant} seed={seed} epoch={epoch} val_NDCG5={score:.4f} stale={stale}", flush=True)\n        if stale >= cfg[\'patience\']:\n            break\n    training_seconds = time.perf_counter()-began\n    model.load_state_dict(torch.load(checkpoint, map_location=device, weights_only=True))\n    validation = evaluate_lm(model, tok, ds, \'val\', variant, seed, cfg, device, run_dir/\'val_predictions.jsonl\')\n    test_began = time.perf_counter()\n    test = evaluate_lm(model, tok, ds, \'test\', variant, seed, cfg, device, run_dir/\'test_predictions.jsonl\')\n    result = dict(model=\'SimpleDyG\', variant=variant, seed=seed, architecture=architecture,\n                  parameters=sum(p.numel() for p in model.parameters()), best_epoch=best_epoch,\n                  epochs_run=len(history), training_seconds=training_seconds,\n                  testing_seconds=time.perf_counter()-test_began, validation=validation, test=test,\n                  tokenizer=tok.vocab)\n    del model, optimizer\n    gc.collect()\n    if device.type == \'cuda\':\n        torch.cuda.empty_cache()\n    return result\n\n\n"""Appended to the scientific core to support isolated workers and full resume."""\nimport argparse\nimport signal\nimport shutil\nimport sqlite3\nimport stat\n\n_RUNTIME = dict(deadline=float(\'inf\'), checkpoint_seconds=300., last_save=0.,\n                pause_after_steps=None, worker_id=None, stop_file=None)\n\n\nclass TrainingPaused(Exception):\n    pass\n\n\ndef runtime_expired():\n    return (time.time() >= _RUNTIME[\'deadline\'] or\n            (_RUNTIME.get(\'stop_file\') is not None and Path(_RUNTIME[\'stop_file\']).exists()))\n\n\ndef check_deadline():\n    if runtime_expired():\n        raise TrainingPaused(\'Session budget reached; restore latest.pt next session.\')\n\n\ndef atomic_torch_save(value, path):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = path.with_name(path.name + \'.tmp\')\n    with temporary.open(\'wb\') as handle:\n        torch.save(value, handle)\n        handle.flush()\n        os.fsync(handle.fileno())\n    temporary.replace(path)\n\n\ndef cpu_state(model):\n    return {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}\n\n\ndef rng_state():\n    state = np.random.get_state()\n    return dict(python=random.getstate(), numpy=dict(name=state[0],\n                keys=torch.from_numpy(state[1].astype(np.int64)), position=state[2],\n                gaussian=state[3], cached=state[4]), torch=torch.get_rng_state(),\n                cuda=torch.cuda.get_rng_state_all() if torch.cuda.is_available() else [])\n\n\ndef restore_rng(state):\n    random.setstate(state[\'python\'])\n    n = state[\'numpy\']\n    np.random.set_state((n[\'name\'], n[\'keys\'].numpy().astype(np.uint32),\n                         n[\'position\'], n[\'gaussian\'], n[\'cached\']))\n    torch.set_rng_state(state[\'torch\'])\n    if torch.cuda.is_available():\n        if len(state[\'cuda\']) != torch.cuda.device_count():\n            raise RuntimeError(\'Checkpoint visible GPU count differs from worker.\')\n        torch.cuda.set_rng_state_all(state[\'cuda\'])\n\n\ndef checkpoint_environment(device):\n    return dict(python=sys.version.split()[0], torch=str(torch.__version__), numpy=np.__version__,\n                accelerator=torch.cuda.get_device_name(device) if device.type == \'cuda\' else \'cpu\')\n\n\ndef new_training_state(identity, total_epochs):\n    return dict(identity=identity, format_version=1, epoch=1, next_batch=0, phase=\'train\',\n                global_step=0, best=-float(\'inf\'), best_epoch=0, stale=0, best_model=None,\n                history=[], losses=[], total_epochs=total_epochs, elapsed_seconds=0.)\n\n\ndef load_training_state(path, identity, model, optimizer, device, scheduler=None, scaler=None):\n    if not Path(path).exists():\n        return None\n    saved = torch.load(path, map_location=\'cpu\', weights_only=True)\n    if saved[\'format_version\'] != 1 or saved[\'identity\'] != identity:\n        raise RuntimeError(\'Checkpoint identity/config mismatch: \' + str(path))\n    if saved[\'environment\'] != checkpoint_environment(device):\n        raise RuntimeError(\'Checkpoint environment differs. Keep the previous Kaggle image/runtime: \' + str(path))\n    model.load_state_dict(saved.pop(\'model\'))\n    optimizer.load_state_dict(saved.pop(\'optimizer\'))\n    if scheduler is not None:\n        scheduler.load_state_dict(saved.pop(\'scheduler\'))\n    if scaler is not None:\n        scaler.load_state_dict(saved.pop(\'scaler\'))\n    restore_rng(saved.pop(\'rng\'))\n    print(f"RESUME {identity}: epoch={saved[\'epoch\']} next_batch={saved[\'next_batch\']} phase={saved[\'phase\']}", flush=True)\n    return saved\n\n\ndef save_training_state(path, state, model, optimizer, device, scheduler=None, scaler=None,\n                        force=False, carry=0., began=None):\n    now = time.monotonic()\n    if not force and now - _RUNTIME[\'last_save\'] < _RUNTIME[\'checkpoint_seconds\']:\n        return\n    saved = dict(state, model=model.state_dict(), optimizer=optimizer.state_dict(),\n                 rng=rng_state(), environment=checkpoint_environment(device))\n    if began is not None:\n        saved[\'elapsed_seconds\'] = carry + time.perf_counter() - began\n    if scheduler is not None:\n        saved[\'scheduler\'] = scheduler.state_dict()\n    if scaler is not None:\n        saved[\'scaler\'] = scaler.state_dict()\n    atomic_torch_save(saved, path)\n    dump_json(Path(path).with_suffix(\'.json\'),\n              dict(identity=state[\'identity\'], epoch=state[\'epoch\'], next_batch=state[\'next_batch\'],\n                   phase=state[\'phase\'], global_step=state[\'global_step\'], saved_at=time.time(),\n                   worker_id=_RUNTIME[\'worker_id\']))\n    _RUNTIME[\'last_save\'] = now\n\n\ndef maybe_pause(state, save):\n    if runtime_expired() or (_RUNTIME[\'pause_after_steps\'] is not None and\n                             state[\'global_step\'] >= _RUNTIME[\'pause_after_steps\']):\n        save(force=True)\n        raise TrainingPaused(\'Checkpoint saved at completed optimizer step.\')\n\n\ndef train_lm(root, ds, variant, seed, cfg, device, run_dir):\n    seed_all(seed)\n    model, tok, architecture = make_lm(root, ds, cfg)\n    model.to(device)\n    sequences = [tok.encode(transform_time(x, variant, x[2], seed, ds[\'spec\'][\'timestamp\']))[-cfg[\'train_block\']:]\n                 for x in ds[\'train\']]\n    compat = cfg[\'protocol\'] == \'author_compat\'\n\n    def collate(rows):\n        padded = torch.full((len(rows), max(map(len, rows))), tok.pad, dtype=torch.long)\n        for i, row in enumerate(rows):\n            padded[i, :len(row)] = torch.tensor(row)\n        return padded\n\n    logical_batch = ds[\'spec\'][\'batch\'] if cfg[\'profile\'] != \'smoke\' else cfg[\'micro_batch\']\n    micro = min(logical_batch, cfg[\'micro_batch\'])\n    if logical_batch % micro:\n        raise ValueError(\'micro_batch must divide logical batch\')\n    generator = torch.Generator().manual_seed(seed)\n    loader = DataLoader(sequences, batch_size=logical_batch, shuffle=True, generator=generator,\n                        collate_fn=collate, drop_last=compat)\n    if not len(loader):\n        raise ValueError(\'No training batches\')\n    optimizer = (AuthorAdamW if compat else torch.optim.AdamW)(model.parameters(),\n                    lr=ds[\'spec\'][\'lr\'], eps=1e-8, weight_decay=0.)\n    scheduler = linear_warmup_decay(optimizer, 0, len(loader) * cfg[\'epochs\'])\n    scaler = torch.amp.GradScaler(\'cuda\', enabled=cfg[\'amp\'] and device.type == \'cuda\')\n    identity = run_id(ds, \'SimpleDyG\', variant, seed, cfg)\n    latest = run_dir / \'latest.pt\'\n    state = load_training_state(latest, identity, model, optimizer, device, scheduler, scaler)\n    if state is None:\n        state = new_training_state(identity, cfg[\'epochs\'])\n        state[\'epoch_generator\'] = generator.get_state()\n    carry, began = state[\'elapsed_seconds\'], time.perf_counter()\n\n    def save(force=False):\n        save_training_state(latest, state, model, optimizer, device, scheduler, scaler,\n                            force=force, carry=carry, began=began)\n\n    # A first checkpoint exists even if the session stops before the first batch.\n    save(force=True)\n    while state[\'epoch\'] <= cfg[\'epochs\'] and state[\'phase\'] != \'done\':\n        if state[\'phase\'] == \'train\':\n            model.train()\n            # Replay only CPU collation up to the saved cursor. This recreates\n            # the original DataLoader shuffle without repeating gradient steps.\n            generator.set_state(state[\'epoch_generator\'])\n            for index, batch in enumerate(loader):\n                if index < state[\'next_batch\']:\n                    continue\n                maybe_pause(state, save)\n                optimizer.zero_grad(set_to_none=True)\n                valid_total = (batch[:, 1:] != tok.pad).sum().item() if not compat else batch[:, 1:].numel()\n                for part in batch.split(micro):\n                    inputs = part.to(device)\n                    labels = inputs.clone()\n                    if not compat:\n                        labels[labels == tok.pad] = -100\n                    weight = (labels[:, 1:] != -100).sum().item() / valid_total\n                    with amp_context(cfg, device):\n                        outputs = model(inputs, labels=labels,\n                                        attention_mask=None if compat else inputs.ne(tok.pad))\n                        loss = outputs[0]\n                    if not torch.isfinite(loss):\n                        raise FloatingPointError(\'Non-finite LM loss\')\n                    scaler.scale(loss * weight).backward()\n                    state[\'losses\'].append(float(loss.detach().cpu()))\n                scaler.unscale_(optimizer)\n                nn.utils.clip_grad_norm_(model.parameters(), 1.)\n                scaler.step(optimizer)\n                scaler.update()\n                scheduler.step()\n                state[\'global_step\'] += 1\n                state[\'next_batch\'] = index + 1\n                save()\n                maybe_pause(state, save)\n            state[\'phase\'] = \'validate\'\n            state[\'next_generator\'] = generator.get_state()\n            save(force=True)\n        if state[\'phase\'] == \'validate\':\n            validation = evaluate_lm(model, tok, ds, \'val\', variant, seed, cfg, device)\n            score = validation[\'ndcg5_author\' if compat else \'ndcg5\']\n            improved = score >= state[\'best\'] if compat else score > state[\'best\'] + 1e-12\n            if improved:\n                state[\'best\'], state[\'best_epoch\'], state[\'stale\'] = score, state[\'epoch\'], 0\n                state[\'best_model\'] = cpu_state(model)\n                atomic_torch_save(state[\'best_model\'], run_dir / \'best.pt\')\n            else:\n                state[\'stale\'] += 1\n            state[\'history\'].append(dict(epoch=state[\'epoch\'], train_loss=float(np.mean(state[\'losses\'])), **validation))\n            dump_json(run_dir / \'history.json\', state[\'history\'])\n            print(f"{ds[\'name\']} SimpleDyG/{variant} seed={seed} epoch={state[\'epoch\']} "\n                  f"val_NDCG5={score:.4f} stale={state[\'stale\']}", flush=True)\n            state[\'epoch\'] += 1\n            state[\'next_batch\'], state[\'losses\'] = 0, []\n            state[\'epoch_generator\'] = state.pop(\'next_generator\')\n            state[\'phase\'] = \'done\' if state[\'stale\'] >= cfg[\'patience\'] or state[\'epoch\'] > cfg[\'epochs\'] else \'train\'\n            save(force=True)\n    model.load_state_dict(state[\'best_model\'])\n    atomic_torch_save(state[\'best_model\'], run_dir / \'best.pt\')\n    training_seconds = carry + time.perf_counter() - began\n    validation = evaluate_lm(model, tok, ds, \'val\', variant, seed, cfg, device, run_dir / \'val_predictions.jsonl\')\n    test_began = time.perf_counter()\n    test = evaluate_lm(model, tok, ds, \'test\', variant, seed, cfg, device, run_dir / \'test_predictions.jsonl\')\n    return dict(model=\'SimpleDyG\', variant=variant, seed=seed, architecture=architecture,\n                parameters=sum(p.numel() for p in model.parameters()), best_epoch=state[\'best_epoch\'],\n                epochs_run=len(state[\'history\']), training_seconds=training_seconds,\n                testing_seconds=time.perf_counter() - test_began, validation=validation, test=test,\n                tokenizer=tok.vocab, optimizer_steps=state[\'global_step\'])\n\n\ndef train_ranker_candidate(key, dim, seed, ds, cfg, device, sampler, classes, run_dir):\n    seed_all(seed)\n    model = TemporalRanker(key, dim, ds, cfg, device, sampler, classes)\n    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)\n    pairs, forbidden = baseline_training_edges(ds, cfg)\n    rng = np.random.default_rng(seed)\n    identity = run_id(ds, key, \'full\', seed, cfg) + f\':dim{dim}\'\n    latest = run_dir / f\'latest_dim{dim}.pt\'\n    checkpoint = run_dir / f\'candidate_dim{dim}.pt\'\n    done = run_dir / f\'candidate_dim{dim}.json\'\n    if done.exists() and checkpoint.exists():\n        receipt = json.loads(done.read_text())\n        if receipt[\'identity\'] != identity:\n            raise RuntimeError(\'Completed baseline candidate identity differs\')\n        model.load_state_dict(torch.load(checkpoint, map_location=device, weights_only=True))\n        return model, receipt[\'validation\'], receipt[\'best_epoch\'], receipt[\'epochs_run\'], checkpoint, receipt[\'training_seconds\']\n    state = load_training_state(latest, identity, model, optimizer, device)\n    if state is None:\n        state = new_training_state(identity, cfg[\'baseline_epochs\'])\n    else:\n        rng.bit_generator.state = state[\'negative_rng\']\n    carry, began = state[\'elapsed_seconds\'], time.perf_counter()\n\n    def save(force=False):\n        state[\'negative_rng\'] = copy.deepcopy(rng.bit_generator.state)\n        save_training_state(latest, state, model, optimizer, device, force=force, carry=carry, began=began)\n\n    save(force=True)\n    while state[\'epoch\'] <= cfg[\'baseline_epochs\'] and state[\'phase\'] != \'done\':\n        if state[\'phase\'] == \'train\':\n            model.train()\n            for index, start in enumerate(range(0, len(pairs), cfg[\'baseline_batch\'])):\n                if index < state[\'next_batch\']:\n                    continue\n                maybe_pause(state, save)\n                batch = pairs[start:start + cfg[\'baseline_batch\']]\n                src = np.array([r[0] for r in batch], dtype=np.int64)\n                dst = np.array([r[1] for r in batch], dtype=np.int64)\n                times = np.array([r[2] for r in batch], dtype=np.float64)\n                negative = []\n                for u, _, _, bin_id in batch:\n                    blocked = forbidden[(u, bin_id)]\n                    if len(blocked) >= len(ds[\'vocab\']):\n                        raise ValueError(\'No negative candidate\')\n                    item = int(rng.integers(1, len(ds[\'vocab\']) + 1))\n                    while item in blocked:\n                        item = int(rng.integers(1, len(ds[\'vocab\']) + 1))\n                    negative.append(item)\n                optimizer.zero_grad(set_to_none=True)\n                negatives = np.array(negative)\n                batch_loss = 0.\n                # Keep the logical event batch and one optimizer step. Smaller\n                # forward batches make GraphMixer\'s 1024-token history fit T4.\n                for sub_start in range(0, len(batch), cfg[\'baseline_micro_batch\']):\n                    part = slice(sub_start, sub_start + cfg[\'baseline_micro_batch\'])\n                    a, b, n, t = src[part], dst[part], negatives[part], times[part]\n                    scores = model(np.concatenate([a, a]), np.concatenate([b, n]), np.concatenate([t, t]))\n                    positive, negative_score = scores.chunk(2)\n                    loss = torch.nn.functional.softplus(negative_score - positive).mean()\n                    if not torch.isfinite(loss):\n                        raise FloatingPointError(\'Non-finite BPR loss\')\n                    weight = len(a) / len(batch)\n                    (loss * weight).backward()\n                    batch_loss += float(loss.detach().cpu()) * weight\n                nn.utils.clip_grad_norm_(model.parameters(), 1.)\n                optimizer.step()\n                state[\'losses\'].append(batch_loss)\n                state[\'global_step\'] += 1\n                state[\'next_batch\'] = index + 1\n                save()\n                maybe_pause(state, save)\n            state[\'phase\'] = \'validate\'\n            save(force=True)\n        if state[\'phase\'] == \'validate\':\n            validation = evaluate_ranker(model, ds, \'val\', cfg, device)\n            score = validation[\'ndcg5\']\n            if score > state[\'best\'] + 1e-12:\n                state[\'best\'], state[\'best_epoch\'], state[\'stale\'] = score, state[\'epoch\'], 0\n                state[\'best_model\'] = cpu_state(model)\n                atomic_torch_save(state[\'best_model\'], checkpoint)\n            else:\n                state[\'stale\'] += 1\n            state[\'history\'].append(dict(epoch=state[\'epoch\'], train_loss=float(np.mean(state[\'losses\'])), **validation))\n            dump_json(run_dir / f\'history_dim{dim}.json\', state[\'history\'])\n            print(f"{ds[\'name\']} {key} d={dim} seed={seed} epoch={state[\'epoch\']} "\n                  f"val_NDCG5={score:.4f} stale={state[\'stale\']}", flush=True)\n            state[\'epoch\'] += 1\n            state[\'next_batch\'], state[\'losses\'] = 0, []\n            state[\'phase\'] = \'done\' if state[\'stale\'] >= cfg[\'patience\'] or state[\'epoch\'] > cfg[\'baseline_epochs\'] else \'train\'\n            save(force=True)\n    model.load_state_dict(state[\'best_model\'])\n    atomic_torch_save(state[\'best_model\'], checkpoint)\n    validation = evaluate_ranker(model, ds, \'val\', cfg, device)\n    seconds = carry + time.perf_counter() - began\n    receipt = dict(identity=identity, validation=validation, best_epoch=state[\'best_epoch\'],\n                   epochs_run=len(state[\'history\']), training_seconds=seconds)\n    dump_json(done, receipt)\n    # Candidate completion is committed; a later interruption can skip its optimizer work.\n    latest.unlink(missing_ok=True)\n    latest.with_suffix(\'.json\').unlink(missing_ok=True)\n    return model, validation, state[\'best_epoch\'], len(state[\'history\']), checkpoint, seconds\n\n\ndef train_ranker(key, seed, ds, cfg, device, dyglib, run_dir):\n    classes, Sampler = load_dyglib(dyglib)\n    sampler = build_sampler(ds, cfg, Sampler)\n    choices = []\n    for dim in cfg[\'baseline_dims\']:\n        check_deadline()\n        model, validation, epoch, epochs_run, checkpoint, seconds = train_ranker_candidate(\n            key, dim, seed, ds, cfg, device, sampler, classes, run_dir)\n        choices.append(dict(dim=dim, validation=validation, best_epoch=epoch,\n                            epochs_run=epochs_run, checkpoint=str(checkpoint), training_seconds=seconds))\n        del model\n        gc.collect()\n        if device.type == \'cuda\':\n            torch.cuda.empty_cache()\n    winner = max(choices, key=lambda c: (c[\'validation\'][\'ndcg5\'], -c[\'dim\']))\n    dump_json(run_dir / \'validation_search.json\', choices)\n    model = TemporalRanker(key, winner[\'dim\'], ds, cfg, device, sampler, classes)\n    model.load_state_dict(torch.load(winner[\'checkpoint\'], map_location=device, weights_only=True))\n    validation = evaluate_ranker(model, ds, \'val\', cfg, device, output=run_dir / \'val_predictions.jsonl\')\n    testing = time.perf_counter()\n    test = evaluate_ranker(model, ds, \'test\', cfg, device, k_choice=validation[\'selected_k\'],\n                           output=run_dir / \'test_predictions.jsonl\')\n    atomic_torch_save(cpu_state(model), run_dir / \'best.pt\')\n    return dict(model=key, variant=\'full\', seed=seed, architecture={\'dim\': winner[\'dim\']},\n                best_epoch=winner[\'best_epoch\'], epochs_run=winner[\'epochs_run\'],\n                parameters=sum(p.numel() for p in model.parameters()), validation=validation, test=test,\n                training_seconds=sum(c[\'training_seconds\'] for c in choices),\n                testing_seconds=time.perf_counter() - testing,\n                feature_policy=\'all-zero node/edge features; dimensions selected on validation\',\n                training_task=\'chronological event BPR, both edge directions; frozen next-bin ranking at evaluation\')\n\n\ndef execute_run(ds, model, variant, seed, cfg, device, output_root, author_root, dyglib_root):\n    identifier = run_id(ds, model, variant, seed, cfg)\n    directory = Path(output_root) / \'runs\' / identifier\n    directory.mkdir(parents=True, exist_ok=True)\n    result_path = directory / \'result.json\'\n    if result_path.exists():\n        previous = json.loads(result_path.read_text())\n        if previous[\'status\'] == \'complete\':\n            return previous\n    dump_json(directory / \'config.json\', cfg)\n    if device.type == \'cuda\':\n        torch.cuda.reset_peak_memory_stats(device)\n    try:\n        if model == \'SimpleDyG\':\n            result = train_lm(author_root, ds, variant, seed, cfg, device, directory)\n        else:\n            result = train_ranker(model, seed, ds, cfg, device, dyglib_root, directory)\n        result.update(status=\'complete\', run_id=identifier, dataset=ds[\'name\'],\n                      protocol=cfg[\'protocol\'], profile=cfg[\'profile\'],\n                      truncated_data=bool(cfg[\'train_limit\'] or cfg[\'eval_limit\'] or cfg[\'edge_limit\']),\n                      peak_gpu_mb=float(torch.cuda.max_memory_allocated(device) / 1e6) if device.type == \'cuda\' else None,\n                      worker_id=_RUNTIME[\'worker_id\'], physical_gpu=os.environ.get(\'CUDA_VISIBLE_DEVICES\'),\n                      checkpoint_format=1)\n    except TrainingPaused as exc:\n        result = dict(status=\'paused\', run_id=identifier, dataset=ds[\'name\'], model=model,\n                      variant=variant, seed=seed, reason=str(exc), protocol=cfg[\'protocol\'], profile=cfg[\'profile\'])\n    except Exception as exc:\n        import traceback\n        (directory / \'error.txt\').write_text(traceback.format_exc(), encoding=\'utf-8\')\n        result = dict(status=\'failed\', run_id=identifier, dataset=ds[\'name\'], model=model,\n                      variant=variant, seed=seed, error=str(exc), protocol=cfg[\'protocol\'], profile=cfg[\'profile\'])\n        print(\'FAILED:\', identifier, str(exc), flush=True)\n    dump_json(result_path, result)\n    if result[\'status\'] == \'complete\':\n        # Delete only this run\'s temporary training states, after result commit.\n        for pattern in [\'latest*.pt\', \'latest*.json\', \'candidate_dim*.pt\']:\n            for path in directory.glob(pattern):\n                if path.resolve().is_relative_to(directory.resolve()):\n                    path.unlink()\n    gc.collect()\n    if device.type == \'cuda\':\n        torch.cuda.empty_cache()\n    return result\n\n\ndef prepare_config(author_root, dyglib_root, cfg):\n    cfg = copy.deepcopy(cfg)\n    files = {}\n    for label, folder in [(\'author\', Path(author_root) / \'models\'), (\'dyglib\', Path(dyglib_root))]:\n        for path in sorted(folder.rglob(\'*.py\')):\n            files[label + \'/\' + path.relative_to(folder).as_posix()] = sha256(path)\n    cfg[\'_source_fingerprint\'] = hashlib.sha256(json.dumps(files, sort_keys=True).encode()).hexdigest()\n    cfg[\'_pipeline_fingerprint\'] = hashlib.sha256(Path(__file__).read_text(encoding=\'utf-8\').encode()).hexdigest()\n    return cfg, files\n\n\ndef dual_plan(datasets, cfg):\n    rows = []\n    for ds in datasets.values():\n        tasks = [(\'SimpleDyG\', variant, seed) for variant in cfg[\'variants\'] for seed in cfg[\'seeds\']]\n        tasks += [(model, \'full\', seed) for model in cfg[\'baseline_models\'] for seed in cfg[\'seeds\']]\n        for model, variant, seed in tasks:\n            rows.append(dict(dataset=ds[\'name\'], model=model, variant=variant, seed=seed,\n                             run_id=run_id(ds, model, variant, seed, cfg)))\n    return rows\n\n\ndef read_records(root, plan):\n    records = []\n    for row in plan:\n        path = Path(root) / \'runs\' / row[\'run_id\'] / \'result.json\'\n        if path.exists():\n            record = json.loads(path.read_text())\n            if record[\'run_id\'] != row[\'run_id\']:\n                raise RuntimeError(\'Result run ID mismatch: \' + str(path))\n            records.append(record)\n        else:\n            records.append(dict(row, status=\'pending\'))\n    return records\n\n\ndef initialize_queue(database, root, plan):\n    with contextlib.closing(sqlite3.connect(database, timeout=60)) as connection, connection:\n        connection.execute(\'PRAGMA journal_mode=WAL\')\n        connection.execute(\'CREATE TABLE IF NOT EXISTS jobs \'\n                           \'(run_id TEXT PRIMARY KEY, ordinal INTEGER, payload TEXT, status TEXT, worker INTEGER, priority INTEGER)\')\n        connection.execute(\'DELETE FROM jobs\')\n        for ordinal, row in enumerate(plan):\n            path = Path(root) / \'runs\' / row[\'run_id\']\n            result = json.loads((path / \'result.json\').read_text()) if (path / \'result.json\').exists() else {}\n            complete = result.get(\'status\') == \'complete\'\n            priority = int(any(path.glob(\'latest*.pt\')) or any(path.glob(\'candidate_dim*.json\')))\n            connection.execute(\'INSERT INTO jobs VALUES (?, ?, ?, ?, NULL, ?)\',\n                               (row[\'run_id\'], ordinal, json.dumps(row), \'complete\' if complete else \'queued\', priority))\n\n\ndef claim_job(database, worker_id):\n    with contextlib.closing(sqlite3.connect(database, timeout=60, isolation_level=None)) as connection:\n        connection.execute(\'BEGIN IMMEDIATE\')\n        row = connection.execute("SELECT run_id, payload FROM jobs WHERE status=\'queued\' "\n                                 \'ORDER BY priority DESC, ordinal LIMIT 1\').fetchone()\n        if row is not None:\n            connection.execute("UPDATE jobs SET status=\'running\', worker=? WHERE run_id=?", (worker_id, row[0]))\n        connection.execute(\'COMMIT\')\n        return json.loads(row[1]) if row else None\n\n\ndef worker_environment(gpu_id=None):\n    env = dict(os.environ)\n    env.update(PYTHONUNBUFFERED=\'1\', PYTHONDONTWRITEBYTECODE=\'1\', USE_TF=\'0\',\n               CUBLAS_WORKSPACE_CONFIG=\':4096:8\', OMP_NUM_THREADS=\'2\', MKL_NUM_THREADS=\'2\')\n    if gpu_id is None:\n        env[\'CUDA_VISIBLE_DEVICES\'] = \'\'\n    else:\n        existing = os.environ.get(\'CUDA_VISIBLE_DEVICES\')\n        devices = existing.split(\',\') if existing else None\n        env[\'CUDA_VISIBLE_DEVICES\'] = devices[gpu_id] if devices else str(gpu_id)\n    return env\n\n\ndef dual_worker(job_file, worker_id):\n    job = json.loads(Path(job_file).read_text())\n    root, cfg = Path(job[\'output\']), job[\'cfg\']\n    torch.set_num_threads(job[\'worker_threads\'])\n    torch.use_deterministic_algorithms(True)\n    if job[\'cpu_workers\']:\n        if cfg[\'profile\'] != \'smoke\':\n            raise RuntimeError(\'CPU workers only validate smoke\')\n        device = torch.device(\'cpu\')\n    else:\n        if torch.cuda.device_count() != 1:\n            raise RuntimeError(\'Each GPU worker must see exactly one GPU\')\n        device = torch.device(\'cuda:0\')\n        torch.cuda.set_device(device)\n    _RUNTIME.update(deadline=job[\'deadline\'], checkpoint_seconds=job[\'checkpoint_seconds\'],\n                    last_save=0., pause_after_steps=job.get(\'pause_after_steps\'), worker_id=worker_id,\n                    stop_file=job[\'stop_file\'])\n    env = checkpoint_environment(device)\n    dump_json(root / f\'worker_{worker_id}.json\', dict(env, pid=os.getpid(), session_id=job[\'session_id\'], state=\'ready\',\n                    physical_gpu=os.environ.get(\'CUDA_VISIBLE_DEVICES\'), ready_at=time.time()))\n    while not Path(job[\'gate\']).exists():\n        if runtime_expired():\n            return\n        time.sleep(.2)\n    datasets = {}\n    while not runtime_expired():\n        row = claim_job(job[\'database\'], worker_id)\n        if row is None:\n            break\n        print(\'CLAIM\', row[\'run_id\'], \'device=\', str(device), \'physical_gpu=\', os.environ.get(\'CUDA_VISIBLE_DEVICES\'), flush=True)\n        if row[\'dataset\'] not in datasets:\n            datasets[row[\'dataset\']] = load_dataset(Path(job[\'author\']), row[\'dataset\'], cfg)\n        result = execute_run(datasets[row[\'dataset\']], row[\'model\'], row[\'variant\'], row[\'seed\'],\n                             cfg, device, root, Path(job[\'author\']), Path(job[\'dyglib\']))\n        with contextlib.closing(sqlite3.connect(job[\'database\'], timeout=60)) as connection, connection:\n            connection.execute(\'UPDATE jobs SET status=? WHERE run_id=?\', (result[\'status\'], row[\'run_id\']))\n        if result[\'status\'] == \'paused\':\n            break\n    dump_json(root / f\'worker_{worker_id}.json\', dict(env, pid=os.getpid(), session_id=job[\'session_id\'], state=\'stopped\',\n                    physical_gpu=os.environ.get(\'CUDA_VISIBLE_DEVICES\'), stopped_at=time.time()))\n\n\ndef create_resume_snapshot(root, destination, include_completed_best=False):\n    """Store checkpoint bytes without expensive recompression; commit atomically."""\n    root, destination = Path(root), Path(destination)\n    temporary = destination.with_name(destination.name + \'.writing\')\n    statuses = {p.parent.name: json.loads(p.read_text()).get(\'status\')\n                for p in (root / \'runs\').glob(\'*/result.json\')}\n    members = []\n    for path in sorted(root.rglob(\'*\')):\n        if not path.is_file() or path.is_symlink():\n            continue\n        if path.resolve() == destination.resolve():\n            continue\n        if path.name == \'SNAPSHOT_MANIFEST.json\':\n            continue\n        relative = path.relative_to(root)\n        if path.name.endswith((\'.tmp\', \'.writing\', \'.lock\', \'.sqlite\', \'.sqlite-wal\', \'.sqlite-shm\')):\n            continue\n        if (path.name.startswith(\'session_\') and not path.name.startswith(\'session_history_\')) or path.name.startswith((\'gate_\', \'stop_\')):\n            continue\n        if path.suffix == \'.pt\' and len(relative.parts) >= 3 and relative.parts[0] == \'runs\':\n            if statuses.get(relative.parts[1]) == \'complete\' and not include_completed_best:\n                continue\n        members.append((path, relative.as_posix()))\n    needed = sum(path.stat().st_size for path, _ in members)\n    if needed > shutil.disk_usage(destination.parent).free:\n        raise RuntimeError(\'Not enough space for resume ZIP; preserve the full Output directory instead.\')\n    inventory = []\n    with zipfile.ZipFile(temporary, \'w\', zipfile.ZIP_STORED, allowZip64=True) as archive:\n        for path, relative in members:\n            archive.write(path, relative)\n            inventory.append(dict(path=relative, bytes=path.stat().st_size))\n        archive.writestr(\'SNAPSHOT_MANIFEST.json\', json.dumps(dict(format_version=1,\n            purpose=\'resume full training states and reuse completed metrics\',\n            include_completed_best=include_completed_best, members=inventory), indent=2))\n    temporary.replace(destination)\n    print(\'Resume ZIP:\', destination, \'| bytes:\', destination.stat().st_size, flush=True)\n    return destination\n\n\ndef restore_dual_output(source, destination):\n    source, destination = Path(source), Path(destination).resolve()\n    destination.mkdir(parents=True, exist_ok=True)\n\n    def safe_target(name):\n        if \'\\\\\' in name or \':\' in name or \'..\' in name.split(\'/\') or name.startswith(\'/\'):\n            raise ValueError(\'Unsafe snapshot path: \' + name)\n        target = (destination / name).resolve()\n        if not target.is_relative_to(destination):\n            raise ValueError(\'Snapshot path outside output\')\n        return target\n\n    def copy_file(target, stream):\n        if target.exists():\n            return\n        target.parent.mkdir(parents=True, exist_ok=True)\n        temporary = target.with_name(target.name + \'.restoring\')\n        with temporary.open(\'wb\') as handle:\n            shutil.copyfileobj(stream, handle, 1024 * 1024)\n        temporary.replace(target)\n\n    def verify_marker(config):\n        if config.get(\'execution_format\') != \'dual_resume_v1\':\n            raise RuntimeError(\'This is an older one-GPU output. Start fresh with the new runtime.\')\n        existing = destination / \'config.json\'\n        if existing.exists() and json.loads(existing.read_text()) != config:\n            raise RuntimeError(\'Output folder has a different experiment configuration\')\n\n    if source.is_dir():\n        verify_marker(json.loads((source / \'config.json\').read_text()))\n        if source.resolve() == destination:\n            return destination\n        if destination.is_relative_to(source.resolve()):\n            raise ValueError(\'Output destination is inside source\')\n        for path in source.rglob(\'*\'):\n            if path.is_symlink():\n                raise ValueError(\'Snapshot source contains a symlink\')\n            if path.is_file() and path.name != \'resume_snapshot.zip\' and not path.name.endswith((\'.tmp\', \'.writing\')):\n                with path.open(\'rb\') as handle:\n                    copy_file(safe_target(path.relative_to(source).as_posix()), handle)\n    else:\n        with zipfile.ZipFile(source) as archive:\n            names = set(archive.namelist())\n            configs = [n for n in names if n.endswith(\'config.json\') and\n                       n[:-len(\'config.json\')] + \'dual_runtime.json\' in names]\n            if len(configs) != 1:\n                raise RuntimeError(\'Choose the new resume_snapshot.zip or its extracted output folder.\')\n            prefix = configs[0][:-len(\'config.json\')]\n            verify_marker(json.loads(archive.read(configs[0])))\n            for info in archive.infolist():\n                if not info.filename.startswith(prefix) or info.is_dir():\n                    continue\n                if stat.S_ISLNK(info.external_attr >> 16):\n                    raise ValueError(\'Snapshot contains a symlink\')\n                target = safe_target(info.filename[len(prefix):])\n                with archive.open(info) as handle:\n                    copy_file(target, handle)\n    return destination\n\n\ndef find_dual_output(search_root):\n    folders = [p.parent for p in Path(search_root).rglob(\'dual_runtime.json\')\n               if (p.parent / \'config.json\').is_file()]\n    if len(folders) == 1:\n        return folders[0]\n    if len(folders) > 1:\n        raise RuntimeError(\'Multiple output versions: set RESUME_FROM explicitly.\')\n    archives = list(Path(search_root).rglob(\'resume_snapshot.zip\'))\n    if len(archives) != 1:\n        raise RuntimeError(\'Add the latest dual-GPU Output or resume_snapshot.zip as Input.\')\n    return archives[0]\n\n\ndef run_dual_pipeline(author_root, dyglib_root, cfg, output_root, gpu_ids=(0, 1),\n                      session_hours=10.5, checkpoint_seconds=300., worker_threads=2,\n                      include_completed_best=False, cpu_workers=0, pause_after_steps=None):\n    from filelock import FileLock\n    root = Path(output_root).resolve()\n    root.mkdir(parents=True, exist_ok=True)\n    with FileLock(str(root / \'coordinator.lock\'), timeout=0):\n        return _run_dual_locked(author_root, dyglib_root, cfg, root, gpu_ids, session_hours,\n                                checkpoint_seconds, worker_threads, include_completed_best,\n                                cpu_workers, pause_after_steps)\n\n\ndef _run_dual_locked(author_root, dyglib_root, cfg, root, gpu_ids, session_hours,\n                     checkpoint_seconds, worker_threads, include_completed_best,\n                     cpu_workers, pause_after_steps):\n    if session_hours <= 0 or checkpoint_seconds <= 0 or worker_threads < 1:\n        raise ValueError(\'Invalid session/checkpoint limits\')\n    if cpu_workers:\n        if cfg[\'profile\'] != \'smoke\' or cpu_workers != 2:\n            raise ValueError(\'CPU validation requires exactly two smoke workers\')\n        devices = [None] * cpu_workers\n    else:\n        if len(gpu_ids) != 2 or len(set(gpu_ids)) != 2 or any(i < 0 or i >= torch.cuda.device_count() for i in gpu_ids):\n            raise RuntimeError(\'Select Kaggle GPU T4 x2; two distinct visible GPUs are required.\')\n        devices = list(gpu_ids)\n    start = time.time()\n    os.environ.setdefault(\'MPLCONFIGDIR\', str(root / \'mpl_cache\'))\n    cfg, sources = prepare_config(Path(author_root), Path(dyglib_root), cfg)\n    existing = root / \'config.json\'\n    if existing.exists() and json.loads(existing.read_text()) != cfg:\n        raise RuntimeError(\'Existing output has different config/core/source. Do not mix experiments.\')\n    environment = dict(python=sys.version, torch=str(torch.__version__), numpy=np.__version__,\n                       pandas=pd.__version__, cuda=torch.version.cuda,\n                       gpu_names=[torch.cuda.get_device_name(i) for i in devices] if not cpu_workers else [\'cpu\', \'cpu\'])\n    if (root / \'environment.json\').exists():\n        previous_environment = json.loads((root / \'environment.json\').read_text())\n        if previous_environment != environment:\n            raise RuntimeError(\'Resume environment differs. Keep the previous PyTorch image and GPU type.\')\n    datasets = {n: load_dataset(author_root, n, cfg) for n in cfg[\'datasets\']}\n    audits = {n: ds[\'audit\'] for n, ds in datasets.items()}\n    if (root / \'data_audit.json\').exists():\n        previous = json.loads((root / \'data_audit.json\').read_text())\n        if any(previous[n][\'files\'] != audits[n][\'files\'] for n in audits):\n            raise RuntimeError(\'Resume data hash differs\')\n    dump_json(root / \'config.json\', cfg)\n    dump_json(root / \'source_hashes.json\', sources)\n    dump_json(root / \'data_audit.json\', audits)\n    dump_json(root / \'dual_runtime.json\', dict(format_version=1, parallelism=\'independent runs\',\n                workers=2, physical_devices=devices, session_hours=session_hours,\n                checkpoint_seconds=checkpoint_seconds, started_at=start,\n                include_completed_best=include_completed_best))\n    dump_json(root / \'environment.json\', environment)\n    (root / \'pip_freeze.txt\').write_text(subprocess.check_output([sys.executable, \'-m\', \'pip\', \'freeze\'], text=True), encoding=\'utf-8\')\n    plan = dual_plan(datasets, cfg)\n    dump_json(root / \'experiment_plan.json\', plan)\n    database = root / \'jobs.sqlite\'\n    initialize_queue(database, root, plan)\n    session = str(time.time_ns())\n    gate, stop_file = root / f\'gate_{session}\', root / f\'stop_{session}\'\n    job_file = root / f\'session_{session}.json\'\n    job = dict(output=str(root), cfg=cfg, author=str(Path(author_root).resolve()),\n               dyglib=str(Path(dyglib_root).resolve()), database=str(database), gate=str(gate),\n               stop_file=str(stop_file), deadline=start + session_hours * 3600,\n               checkpoint_seconds=checkpoint_seconds, worker_threads=worker_threads,\n               cpu_workers=cpu_workers, pause_after_steps=pause_after_steps, session_id=session)\n    dump_json(job_file, job)\n    workers, handles, offsets = [], [], [0, 0]\n    last_export, previous_status = 0., None\n    session_history = dict(started_at=start, devices=devices, worker_pids=[], gate_opened_at=None,\n                           stopped_at=None, status=\'running\', cfg_fingerprint=cfg[\'_pipeline_fingerprint\'])\n\n    def show_logs():\n        for index in range(2):\n            path = root / f\'worker_{index}.log\'\n            if path.exists():\n                with path.open(\'r\', encoding=\'utf-8\', errors=\'replace\') as handle:\n                    handle.seek(offsets[index])\n                    lines = handle.read()\n                    offsets[index] = handle.tell()\n                for line in lines.splitlines():\n                    print(f\'[worker {index} / device {devices[index]}] {line}\', flush=True)\n\n    try:\n        for index, device in enumerate(devices):\n            log = (root / f\'worker_{index}.log\').open(\'w\', encoding=\'utf-8\')\n            handles.append(log)\n            flags = dict(creationflags=subprocess.CREATE_NO_WINDOW) if os.name == \'nt\' else {}\n            worker = subprocess.Popen([sys.executable, \'-u\', str(Path(__file__).resolve()),\n                                        \'--worker-job\', str(job_file), \'--worker-id\', str(index)],\n                                       env=worker_environment(device), stdout=log, stderr=subprocess.STDOUT, **flags)\n            workers.append(worker)\n            session_history[\'worker_pids\'].append(worker.pid)\n        ready_deadline = min(job[\'deadline\'], time.time() + 180)\n        while True:\n            show_logs()\n            ready = [json.loads((root / f\'worker_{i}.json\').read_text())\n                     if (root / f\'worker_{i}.json\').exists() else {} for i in range(2)]\n            if all(r.get(\'session_id\') == session and r.get(\'state\') == \'ready\' for r in ready):\n                gate.touch()\n                session_history[\'actual_worker_pids\'] = [r[\'pid\'] for r in ready]\n                session_history[\'gate_opened_at\'] = time.time()\n                print(\'Two workers READY:\', ready, flush=True)\n                break\n            if any(w.poll() is not None for w in workers) or time.time() >= ready_deadline:\n                raise RuntimeError(\'Worker startup failed; see worker_0.log and worker_1.log\')\n            time.sleep(.5)\n        while any(w.poll() is None for w in workers):\n            show_logs()\n            records = read_records(root, plan)\n            statuses = tuple((r[\'run_id\'], r[\'status\']) for r in records)\n            if statuses != previous_status and time.time() - last_export > 10:\n                export_results(records, cfg, root, make_archive=False)\n                previous_status, last_export = statuses, time.time()\n                counts = dict(Counter(r[\'status\'] for r in records))\n                print(\'Progress:\', counts, \'| total:\', len(plan), flush=True)\n            time.sleep(.5 if cfg[\'profile\'] == \'smoke\' else 3)\n    except BaseException:\n        stop_file.touch()\n        # Workers save on the next batch/evaluation boundary. Preserve existing\n        # states if a worker hangs; never wait until the external 12h deadline.\n        for worker in workers:\n            try:\n                worker.wait(timeout=60)\n            except subprocess.TimeoutExpired:\n                worker.terminate()\n                worker.wait(timeout=15)\n        session_history[\'status\'] = \'interrupted\'\n        raise\n    finally:\n        for handle in handles:\n            handle.close()\n        show_logs()\n        session_history[\'stopped_at\'] = time.time()\n        if session_history[\'status\'] == \'running\':\n            session_history[\'status\'] = \'finished\'\n        dump_json(root / f\'session_history_{session}.json\', session_history)\n        records = read_records(root, plan)\n        export_results(records, cfg, root, make_archive=False)\n        snapshot = create_resume_snapshot(root, root / \'resume_snapshot.zip\', include_completed_best)\n    failed_workers = [w.returncode for w in workers if w.returncode != 0]\n    if failed_workers or any(r[\'status\'] == \'failed\' for r in records):\n        raise RuntimeError(\'Some runs/workers failed; checkpoints and ZIP retained. Inspect worker logs/error.txt.\')\n    df, summary, _ = export_results(records, cfg, root, make_archive=False)\n    print(\'SESSION SAVED:\', dict(Counter(r[\'status\'] for r in records)), \'| resume ZIP:\', snapshot, flush=True)\n    return df, summary, snapshot\n\n\nif __name__ == \'__main__\':\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\'--worker-job\', required=True)\n    parser.add_argument(\'--worker-id\', type=int, required=True)\n    arguments = parser.parse_args()\n    dual_worker(arguments.worker_job, arguments.worker_id)\n'
RUNTIME_PATH = WORK / "simpledyg_dual_runtime.py"
RUNTIME_PATH.write_text(DUAL_RUNTIME_SOURCE, encoding="utf-8", newline="\n")
assert hashlib.sha256(RUNTIME_PATH.read_text(encoding="utf-8").encode()).hexdigest() == "e6e6142856ea68e46feba431bdfbf9748380818caffaa37c604745af7d972ab8"
spec = importlib.util.spec_from_file_location("simpledyg_dual_runtime", RUNTIME_PATH)
dual = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = dual
spec.loader.exec_module(dual)
sha256, AUTHOR_COMMIT, DYGLIB_COMMIT = dual.sha256, dual.AUTHOR_COMMIT, dual.DYGLIB_COMMIT
print("Runtime SHA-256:", "e6e6142856ea68e46feba431bdfbf9748380818caffaa37c604745af7d972ab8")


## 3. Source và dữ liệu được pin

In [ ]:
# Ưu tiên bundle/attached repo; chỉ clone đúng commit khi chưa có dữ liệu.
INPUT = Path("/kaggle/input")
bundles = [Path(INPUT_BUNDLE_OVERRIDE)] if INPUT_BUNDLE_OVERRIDE else (list(INPUT.rglob("SimpleDyG_Kaggle_2GPU_Inputs.zip")) if INPUT.exists() else [])
EXTRACTED = WORK/"simpledyg_inputs"
if bundles:
    with zipfile.ZipFile(bundles[0]) as archive:
        for info in archive.infolist():
            target = (EXTRACTED/info.filename).resolve()
            if not target.is_relative_to(EXTRACTED.resolve()):
                raise ValueError("ZIP chứa đường dẫn vượt ngoài thư mục input")
        archive.extractall(EXTRACTED)
    manifest = json.loads((EXTRACTED/"SOURCE_MANIFEST.json").read_text())
    for relative, expected in manifest["sha256"].items():
        if sha256(EXTRACTED/relative) != expected:
            raise ValueError("Input checksum mismatch: "+relative)
    assert manifest["author_commit"] == AUTHOR_COMMIT and manifest["dyglib_commit"] == DYGLIB_COMMIT

def obtain_repo(label, override, url, commit):
    if override:
        root = Path(override)
    elif (EXTRACTED/label).exists():
        root = EXTRACTED/label
    else:
        root = None
        if INPUT.exists():
            if label == "SimpleDyG":
                for p in INPUT.rglob("train.link_prediction"):
                    if p.parent.name == "12" and p.parent.parent.name == "UCI_13":
                        root = p.parents[3]
                        break
            else:
                found = list(INPUT.rglob("TGAT.py"))
                root = found[0].parents[1] if found else None
        if root is None:
            root = WORK/label
            if not root.exists():
                if OFFLINE:
                    raise FileNotFoundError(f"Offline: không tìm thấy {label}. Add Input bundle được cung cấp.")
                subprocess.run(["git", "clone", "--no-checkout", url, str(root)], check=True)
                subprocess.run(["git", "-C", str(root), "checkout", commit], check=True)
    if not root.is_dir():
        raise FileNotFoundError(root)
    if (root/".git").exists():
        revision = subprocess.check_output(["git", "-C", str(root), "rev-parse", "HEAD"], text=True).strip()
        if revision != commit:
            raise ValueError(f"{label} commit khác pin: {revision}. Dùng bundle hoặc clone đúng commit {commit}.")
    print(label, root)
    return root

AUTHOR_ROOT = obtain_repo("SimpleDyG", AUTHOR_ROOT_OVERRIDE, "https://github.com/YuxiaWu/SimpleDyG.git", AUTHOR_COMMIT)
DYGLIB_ROOT = obtain_repo("DyGLib", DYGLIB_ROOT_OVERRIDE, "https://github.com/yule-BUAA/DyGLib.git", DYGLIB_COMMIT)
for root, required in [(AUTHOR_ROOT, ["models/modeling_gpt2.py", "resources/UCI_13/12/train.link_prediction"]),
                       (DYGLIB_ROOT, ["models/TGAT.py", "models/GraphMixer.py", "utils/utils.py"])]:
    for relative in required:
        if not (root/relative).is_file():
            raise FileNotFoundError(root/relative)


## 4. Cấu hình mới hoặc khôi phục phiên trước

In [ ]:
"""Prepare a source tree with the exact saved inventory before resuming.

Kept outside the training runtime so existing run/checkpoint identities stay valid.
Extra Python files in a full clone are excluded only after every saved file has
passed its original checksum. No saved fingerprint or checkpoint is rewritten.
"""
import hashlib
import json
from pathlib import Path
import shutil
import tempfile


def align_resume_sources(runtime, author_root, dyglib_root, saved_config, output_root, work_root):
    output_root, work_root = Path(output_root), Path(work_root)
    expected = json.loads((output_root / 'source_hashes.json').read_text(encoding='utf-8'))
    fingerprint = hashlib.sha256(json.dumps(expected, sort_keys=True).encode()).hexdigest()
    if fingerprint != saved_config.get('_source_fingerprint'):
        raise RuntimeError('Saved source manifest does not match config; choose the intact previous Output.')
    science_config = {k: v for k, v in saved_config.items() if not k.startswith('_')}
    current_config, actual = runtime.prepare_config(author_root, dyglib_root, science_config)
    if current_config['_pipeline_fingerprint'] != saved_config.get('_pipeline_fingerprint'):
        raise RuntimeError('Training runtime differs from the saved run. Use the notebook with its original runtime.')

    roots = {'author': Path(author_root) / 'models', 'dyglib': Path(dyglib_root)}

    def checked_path(root, relative):
        if not relative or '\\' in relative or ':' in relative or '..' in relative.split('/') or relative.startswith('/'):
            raise ValueError('Unsafe saved source/data path: ' + relative)
        path = (root / relative).resolve()
        if not path.is_relative_to(root.resolve()):
            raise ValueError('Saved source/data path leaves its root: ' + relative)
        if not path.is_file():
            raise FileNotFoundError('Saved source/data file is missing: ' + str(path))
        return path

    source_paths = {}
    labels = set()
    for name, wanted in expected.items():
        label, separator, relative = name.partition('/')
        if not separator or label not in roots or not relative.endswith('.py'):
            raise ValueError('Unsupported saved source entry: ' + name)
        path = checked_path(roots[label], relative)
        if runtime.sha256(path) != wanted:
            raise RuntimeError('Saved source checksum differs: ' + name + '. Attach the original source bundle.')
        source_paths[name] = path
        labels.add(label)
    if labels != {'author', 'dyglib'}:
        raise RuntimeError('Saved source manifest must include author and DyGLib files.')
    extras = sorted(set(actual) - set(expected))
    report = dict(saved_source_files=len(expected), verified_source_files=len(source_paths),
                  extra_files_excluded=extras, source_fingerprint=fingerprint,
                  runtime_fingerprint=current_config['_pipeline_fingerprint'], aligned=False)
    if actual == expected:
        if current_config != saved_config:
            raise RuntimeError('Saved experiment config differs even with matching source.')
        print('Resume source checksums verified:', len(expected), 'files; original inventory already matches.')
        return Path(author_root), Path(dyglib_root), report

    # Copy only the original source inventory into a fresh workspace directory.
    # Fresh staging avoids changing or deleting the full clone or the Input.
    work_root.mkdir(parents=True, exist_ok=True)
    stage = Path(tempfile.mkdtemp(prefix='resume_verified_sources_', dir=work_root))
    author_view, dyglib_view = stage / 'SimpleDyG', stage / 'DyGLib'
    targets = {'author': author_view / 'models', 'dyglib': dyglib_view}
    for name, path in source_paths.items():
        label, relative = name.split('/', 1)
        target = targets[label] / relative
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copyfile(path, target)

    # The author root also holds dataset/vocab/raw graph files used by the core.
    # Reproduce that tree from the saved data audit and check its original bytes.
    audits = json.loads((output_root / 'data_audit.json').read_text(encoding='utf-8'))
    data_files = {}
    for dataset in science_config['datasets']:
        for relative, wanted in audits[dataset]['files'].items():
            if relative in data_files and data_files[relative] != wanted:
                raise RuntimeError('Conflicting saved dataset hashes: ' + relative)
            data_files[relative] = wanted
    for relative, wanted in data_files.items():
        if relative.split('/')[0] not in {'resources', 'vocabs', 'all_data'}:
            raise ValueError('Unsupported saved dataset entry: ' + relative)
        path = checked_path(Path(author_root), relative)
        if runtime.sha256(path) != wanted:
            raise RuntimeError('Saved dataset checksum differs: ' + relative + '. Attach the original dataset bundle.')
        target = author_view / relative
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copyfile(path, target)
    verified_config, verified_files = runtime.prepare_config(author_view, dyglib_view, science_config)
    if verified_files != expected or verified_config != saved_config:
        raise RuntimeError('Verified source view could not reproduce the original experiment identity.')
    report.update(aligned=True, verified_data_files=len(data_files),
                  author_view=str(author_view), dyglib_view=str(dyglib_view))
    print('RESUME source aligned:', len(expected), 'verified Python files;',
          len(extras), 'extra files excluded; original runtime/config identity preserved.')
    return author_view, dyglib_view, report


OUTPUT = WORK / "simpledyg_2gpu_results"
if RESUME:
    previous = Path(RESUME_FROM) if RESUME_FROM else dual.find_dual_output(INPUT)
    dual.restore_dual_output(previous, OUTPUT)
    saved = json.loads((OUTPUT / "config.json").read_text())
    AUTHOR_ROOT, DYGLIB_ROOT, source_alignment_report = align_resume_sources(
        dual, AUTHOR_ROOT, DYGLIB_ROOT, saved, OUTPUT, WORK)
    (OUTPUT / "resume_source_alignment.json").write_text(
        json.dumps(source_alignment_report, indent=2), encoding="utf-8")
    # Full config comes from the previous output; scheduling options remain editable.
    CFG = {k: v for k, v in saved.items() if not k.startswith("_")}
    print("RESUME source:", previous)
else:
    if (OUTPUT / "config.json").exists():
        raise RuntimeError("Output đã có thí nghiệm. Đặt RESUME=True để tiếp tục; dùng phiên mới để chạy từ đầu.")
    overrides = dict(variants=VARIANTS, baseline_models=BASELINE_MODELS, export_checkpoints=True)
    if PROTOCOL_OVERRIDE is not None:
        overrides["protocol"] = PROTOCOL_OVERRIDE
    if DATASETS_OVERRIDE is not None:
        overrides["datasets"] = DATASETS_OVERRIDE
    if SEEDS_OVERRIDE is not None:
        overrides["seeds"] = SEEDS_OVERRIDE
    overrides.update(EXTRA_CONFIG)
    CFG = dual.make_config(PROFILE, **overrides)
print(json.dumps(CFG, indent=2))
for name in CFG["datasets"]:
    audit = dual.load_dataset(AUTHOR_ROOT, name, CFG)["audit"]
    print(name, json.dumps(audit["splits"], indent=2))
print("Planned runs:", len(CFG["datasets"])*len(CFG["seeds"])*(len(CFG["variants"])+len(CFG["baseline_models"])))


## 5. Chạy song song, checkpoint và tạm dừng

In [ ]:
# Hai tiến trình độc lập; mỗi GPU xử lý một run, không nhân đôi batch size.
# Runtime ghi latest.pt đầy đủ, best.pt riêng và kết quả bằng atomic replace.
# Smoke có thể kiểm tra cùng đường chạy hai tiến trình trên CPU khi không có GPU.
cpu_check = 2 if CFG["profile"] == "smoke" and not torch.cuda.is_available() else 0
results, summary, resume_archive = dual.run_dual_pipeline(
    AUTHOR_ROOT, DYGLIB_ROOT, CFG, OUTPUT, gpu_ids=GPU_IDS,
    session_hours=SESSION_HOURS, checkpoint_seconds=CHECKPOINT_SECONDS,
    worker_threads=WORKER_CPU_THREADS, include_completed_best=SNAPSHOT_INCLUDE_COMPLETED_BEST,
    cpu_workers=cpu_check)
display(summary)


## 6. Kết quả và gói chạy tiếp

In [ ]:
from IPython.display import FileLink, Image, display
display(FileLink(str(resume_archive)))
display(results)
for filename in ["ndcg5.png", "jaccard_at5.png"]:
    if (OUTPUT/filename).exists():
        display(Image(filename=str(OUTPUT/filename)))
display(pd.read_csv(OUTPUT/"paper_comparison.csv"))
display(pd.read_csv(OUTPUT/"ablation_paired_deltas.csv"))
records = json.loads((OUTPUT/"run_manifest.json").read_text())
print("Run status:", dict(dual.Counter(r["status"] for r in records)))
print("Lưu Version/Output và resume_snapshot.zip. Phiên sau Add Input Output mới nhất, đặt RESUME=True.")
print("ZIP resume mặc định gồm checkpoint đầy đủ của run đang dở và best.pt của các run hoàn thành.")


## Phiên tiếp theo

1. Lưu Output và `simpledyg_2gpu_results/resume_snapshot.zip` của phiên hiện tại.
2. Add Input **Output mới nhất** hoặc ZIP snapshot mới nhất; bỏ các snapshot cũ khỏi Input.
3. Giữ ZIP source/dataset cũ và image PyTorch/GPU type cũ, bật GPU T4 x2.
4. Đặt `RESUME=True`; để `RESUME_FROM=""` nếu chỉ có một snapshot đúng.
5. Run All. Log `RESUME ... epoch=... next_batch=... phase=...` xác nhận đọc checkpoint;
   các run hoàn thành sẽ được bỏ qua. Chỉ chỉnh SESSION_HOURS/CHECKPOINT_SECONDS khi chia phiên.

Checkpoint tương thích source/data/config/environment cũ được kiểm tra trước khi dùng.
Khi source lấy từ full clone có thêm file Python, notebook dựng thư mục source đúng
manifest đã lưu sau khi kiểm tra checksum từng file; giữ nguyên runtime và run ID.
Đổi seed, epoch, history, model, dimension, micro-batch hoặc protocol tạo thí nghiệm khác.
`SNAPSHOT_INCLUDE_COMPLETED_BEST=True` mặc định giữ cả trọng số hoàn thành trong ZIP.
Có thể đặt False để giảm dung lượng ZIP resume; khi đó cần giữ riêng Output gốc chứa best.pt.
Giữ toàn bộ Output để phân tích prediction và đánh giá lại từng checkpoint.

Nếu có run failed, đọc worker_0.log/worker_1.log và runs/.../error.txt; các trạng thái
paused/pending thể hiện lịch còn thiếu. Chỉ báo kết quả theo số seed thực tế đã hoàn thành.
